In [ ]:
selected_model = 'name-of-best-bytestorm-model'
tracking_type = 'TT0.7-TB3-MT0.8-BB21'
our_work_name = 'ByteStorm'
setup = 'setup-name'

# rolling months
years = range(1980, 2020)
months = "rolling"

# lat and lon ranges of the domain
lon_range = [0, 360]
lat_range = [-70, 70]

# maximum distance to consider true the match
max_track_distance_matching = 300.0

# whether to store the figures or not
save_figures = True
compute_gif = False

# Get all the basins on which we compute metrics
basins = ['WNP', 'ENP', 'NI', 'SI', 'SP', 'NATL']
excluded_basins = ['SA', 'MED']

# output file extention
plot_kwargs = dict(dpi=100, bbox_inches='tight')
file_ext = 'pdf'

In [ ]:
from cartopy.mpl.ticker import (LongitudeFormatter, LatitudeFormatter)
from math import atan2, radians, degrees, sin, cos
import matplotlib.transforms as transforms
import matplotlib.animation as animation
from haversine import haversine_vector
import matplotlib.ticker as mticker
from geopy.distance import geodesic
from scipy.stats import pearsonr
from IPython.display import HTML
import matplotlib.pyplot as plt
import cartopy.feature as cf
import cartopy.crs as ccrs
import seaborn as sns
import xarray as xr
import pandas as pd
import numpy as np
import sys
import os

sys.path.append('../resources/library')
from tropical_cyclone.cyclone import (
    filter_basin,
    filter_tracks,
    compute_pod_and_far,
    plot_storm_track_density,
    compute_storm_transits,
    signif,
    plot_pod_and_far,
    remove_tcs_above_x_lat,
    remove_tcs_originated_on_land,
    plot_tracks,
    remove_short_tracks,
    convert_coords_to_format,
    plot_pod_and_far_on_axis,
    bearing,
    track_smoothness,
    clean_tracks,
)
from tropical_cyclone.utils import build_rolling_months
import dynamicopy

import warnings
warnings.filterwarnings('ignore')

%load_ext autoreload
%autoreload 2

In [ ]:
# main dirs
data_src = 'PATH-TO/data'
bourdin_trackers_src = 'PATH-TO/zenodo_bourdin/trackers'
inference_src = os.path.join(data_src, 'inference-global')

# trackers source
ibtracs_src = os.path.join(data_src, 'ibtracs', 'filtered', 'global_ibtracs_main-tracks_6h_1980-2022_TS-NR-ET-MX-SS-DS.csv')
owz_src = os.path.join(bourdin_trackers_src, 'OWZ', 'tracks', 'ERA5_OWZ.csv')
track_src = os.path.join(bourdin_trackers_src, 'TRACK', 'ERA5_TRACK.csv')
uz_src = os.path.join(bourdin_trackers_src, 'UZ', 'tracks', 'ERA5_UZ.csv')
cnrm_src = os.path.join(bourdin_trackers_src, 'CNRM', 'tracks_vor15_res5_t1_pt-1_pw5_rel25.csv')

# our model
bytestorm_src = os.path.join(inference_src, f'{selected_model}/tracking_{tracking_type}-CLEAN.csv')

# directory to store images

out_dir = f'./notebooks/tracking/images/{setup}/2020-2023_{selected_model}'
if tracking_type is not None:
    out_dir += f'-{tracking_type}'
if months == "rolling":
    out_dir += f'-TEST-ROLLING-MONTHS'

In [ ]:
os.makedirs(out_dir, exist_ok=True)
out_dir

In [ ]:
INFO = {
    'ibtracs': {
        'color': 'tab:blue', 
        'label': 'IBTrACS', 
    }, 
    'owz': {
        'color': 'tab:pink', 
        'label': 'OWZ', 
    }, 
    'track': {
        'color': 'tab:green', 
        'label': 'TRACK', 
    }, 
    'uz': {
        'color': 'tab:orange', 
        'label': 'UZ', 
    }, 
    'cnrm': {
        'color': 'tab:purple', 
        'label': 'CNRM', 
    }, 
    'bytestorm': {
        'color': 'tab:red', 
        'label': f'{our_work_name}', 
    }, 
}

In [ ]:
ibtracs = pd.read_csv(ibtracs_src, index_col=0).rename(columns={'ISO_TIME':'time','LAT':'lat','LON':'lon','SID':'track_id','BASIN':'basin'})[['time','lat','lon','track_id','basin','NATURE','WMO_WIND']]
owz = pd.read_csv(owz_src).rename(columns={' year':'year', ' month':'month', ' day':'day', ' hour':'hour',' lon':'lon',' lat':'lat'})
track = pd.read_csv(track_src, index_col=0)
uz = pd.read_csv(uz_src).rename(columns={' year':'year', ' month':'month', ' day':'day', ' hour':'hour',' lon':'lon',' lat':'lat'})
cnrm = pd.read_csv(cnrm_src).rename(columns={'Longitude':'lon', 'Latitude':'lat', 'Date':'time', 'ID':'track_id'})
bytestorm = pd.read_csv(bytestorm_src, index_col=0)
# Load Bourdin IBTrACS from dynamicopy
ibtracs_bo = dynamicopy.load_ibtracs()

In [ ]:
# convert iso_time to pandas datetime format
ibtracs['time'] = pd.to_datetime(ibtracs['time'])
owz['time'] = pd.to_datetime(owz[['year','month','day','hour']])
track['time'] = pd.to_datetime(track['time'])
uz['time'] = pd.to_datetime(uz[['year','month','day','hour']])
cnrm['time'] = pd.to_datetime(cnrm['time'])
bytestorm['time'] = pd.to_datetime(bytestorm['time'])
ibtracs_bo['time'] = pd.to_datetime(ibtracs_bo['time'])

In [ ]:
# Add basins information with dynamicopy
ibtracs['basin'] = dynamicopy.get_basin(ibtracs['lon'], ibtracs['lat'])
owz['basin'] = dynamicopy.get_basin(owz['lon'], owz['lat'])
track['basin'] = dynamicopy.get_basin(track['lon'], track['lat'])
uz['basin'] = dynamicopy.get_basin(uz['lon'], uz['lat'])
cnrm['basin'] = dynamicopy.get_basin(cnrm['lon'], cnrm['lat'])
bytestorm['basin'] = dynamicopy.get_basin(bytestorm['lon'], bytestorm['lat'])
ibtracs_bo['basin'] = dynamicopy.get_basin(ibtracs_bo['lon'], ibtracs_bo['lat'])

In [ ]:
if months == "rolling":
    year_and_month, years_dict = build_rolling_months(years[0], years[-1])
    test_years_dict = dict()
    for y,m in year_and_month:
        if y not in test_years_dict.keys():
            test_years_dict[y] = []
        test_years_dict[y] += [m]
    test_years_dict
    avail_dates = []
    dates = ibtracs['time'].unique()
    for date in dates:
        if date.year not in test_years_dict.keys(): continue
        if date.month in test_years_dict[date.year]:
            avail_dates.append(date)
    # for year in test_years_dict.keys():
    #     dates 
    #     dates = [date for date in dates if dates.year in test_years_dict.keys() and date.month in test_years_dict.keys()]
    dates = avail_dates
else:
    # Get the dates on which we perform evaluation
    dates = ibtracs['time'].unique()
    dates = [date for date in dates if date.year in years and date.month in months]
len(dates), len(ibtracs['time'].unique())

In [ ]:
# get intersection of all dates to be sure that each timesteps has been seen by EVERY tracker
# filter the tracks according to the procedure.
ibtracs = ibtracs[ibtracs['time'].isin(dates)]
owz = owz[owz['time'].isin(dates)]
track = track[track['time'].isin(dates)]
uz = uz[uz['time'].isin(dates)]
cnrm = cnrm[cnrm['time'].isin(dates)]
bytestorm = bytestorm[bytestorm['time'].isin(dates)]
ibtracs_bo = ibtracs_bo[ibtracs_bo['time'].isin(dates)]

In [ ]:
ibtracs = remove_short_tracks(ibtracs, min_track_count=12)
owz = remove_short_tracks(owz, min_track_count=12)
track = remove_short_tracks(track, min_track_count=12)
uz = remove_short_tracks(uz, min_track_count=12)
cnrm = remove_short_tracks(cnrm, min_track_count=12)
bytestorm = remove_short_tracks(bytestorm, min_track_count=12)
# ibtracs_bo = remove_short_tracks(ibtracs_bo, min_track_count=4)

In [ ]:
# Convert the coordinates to the [-180,180] format
coords_format = '[-180,180]'
ibtracs = convert_coords_to_format(ibtracs, format=coords_format)
ibtracs_bo = convert_coords_to_format(ibtracs_bo, format=coords_format)
owz = convert_coords_to_format(owz, format=coords_format)
track = convert_coords_to_format(track, format=coords_format)
uz = convert_coords_to_format(uz, format=coords_format)
cnrm = convert_coords_to_format(cnrm, format=coords_format)
bytestorm = convert_coords_to_format(bytestorm, format=coords_format)

# POD and FAR

In [ ]:
owz_matches, owz_res = compute_pod_and_far(dynamicopy, owz, 'owz', ibtracs, max_track_distance_matching, print_results=False)
track_matches, track_res = compute_pod_and_far(dynamicopy, track, 'track', ibtracs, max_track_distance_matching, print_results=False)
uz_matches, uz_res = compute_pod_and_far(dynamicopy, uz, 'uz', ibtracs, max_track_distance_matching, print_results=False)
cnrm_matches, cnrm_res = compute_pod_and_far(dynamicopy, cnrm, 'cnrm', ibtracs, max_track_distance_matching, print_results=False)
bytestorm_matches, bytestorm_res = compute_pod_and_far(dynamicopy, bytestorm, our_work_name, ibtracs, max_track_distance_matching, print_results=False)
algo_results = pd.concat([owz_res, track_res, uz_res, cnrm_res, bytestorm_res]).reset_index(drop=True)
algo_results['model'] = selected_model
algo_results['pod'] = algo_results['pod'] * 100
algo_results['far'] = algo_results['far'] * 100
outfile = os.path.join(out_dir, f'pod_far_ibtracs.{file_ext}') if save_figures else None
plot_pod_and_far(algo_results, label=f' ({years[0]}-{years[-1]})', outfile=outfile)

In [ ]:
owz_matches_bo, owz_res_bo = compute_pod_and_far(dynamicopy, owz, 'owz', ibtracs_bo, max_track_distance_matching, print_results=False)
track_matches_bo, track_res_bo = compute_pod_and_far(dynamicopy, track, 'track', ibtracs_bo, max_track_distance_matching, print_results=False)
uz_matches_bo, uz_res_bo = compute_pod_and_far(dynamicopy, uz, 'uz', ibtracs_bo, max_track_distance_matching, print_results=False)
cnrm_matches_bo, cnrm_res_bo = compute_pod_and_far(dynamicopy, cnrm, 'cnrm', ibtracs_bo, max_track_distance_matching, print_results=False)
bytestorm_matches_bo, bytestorm_res_bo = compute_pod_and_far(dynamicopy, bytestorm, our_work_name, ibtracs_bo, max_track_distance_matching, print_results=False)
algo_results_bo = pd.concat([owz_res_bo, track_res_bo, uz_res_bo, cnrm_res_bo, bytestorm_res_bo]).reset_index(drop=True)
algo_results_bo['model'] = selected_model
algo_results_bo['pod'] = algo_results_bo['pod'] * 100
algo_results_bo['far'] = algo_results_bo['far'] * 100
outfile = os.path.join(out_dir, f'pod_far_bourdin.{file_ext}') if save_figures else None
plot_pod_and_far(algo_results_bo, label=f' ({years[0]}-{years[-1]})', outfile=outfile)

In [ ]:
def compute_pod_and_far_by_basin(
        dynamicopy,
        algo: str,
        dets: pd.DataFrame,
        obss: pd.DataFrame,
        max_track_distance_matching: int,
        basin: str,
        by_100: bool = True,
    ):
    dets_basin = dets[dets["basin"]==basin]
    obss_basin = obss[obss["basin"]==basin]
    if len(dets_basin) == 0:
        # we have no TCs within this basin
        # therefore we have to prepare an empty pd.DataFrame
        matches = pd.DataFrame(data={
            f'id_{algo}':[], 'id_ibtracs':[], 'matching_timesteps':[], 'dist':[],    
        })
        res = pd.DataFrame(data={
            'algo':[algo], 'hits':[0], 'misses':[0], 'false alarms':[0], 'pod':[0], 'far':[0],
        })
    else:
        matches, res = compute_pod_and_far(dynamicopy, dets_basin, algo, obss_basin, max_track_distance_matching, print_results=False)
    if by_100:
        res['pod'] = res['pod'] * 100
        res['far'] = res['far'] * 100
        res['sr'] = res['sr'] * 100
        res['csi'] = res['csi'] * 100
    return matches, res

In [ ]:
# Dictionary for POD-FAR results using our IBTrACS
owz_matches_dict_ib, owz_res_dict_ib = {}, {}
track_matches_dict_ib, track_res_dict_ib = {}, {}
uz_matches_dict_ib, uz_res_dict_ib = {}, {}
cnrm_matches_dict_ib, cnrm_res_dict_ib = {}, {}
bytestorm_matches_dict_ib, bytestorm_res_dict_ib = {}, {}
# Dictionary for POD-FAR results using Bourdin IBTrACS
owz_matches_dict_bo, owz_res_dict_bo = {}, {}
track_matches_dict_bo, track_res_dict_bo = {}, {}
uz_matches_dict_bo, uz_res_dict_bo = {}, {}
cnrm_matches_dict_bo, cnrm_res_dict_bo = {}, {}
bytestorm_matches_dict_bo, bytestorm_res_dict_bo = {}, {}

In [ ]:
# Compute POD and FAR for each basin and for each TC Tracker
for basin in basins:
    owz_matches_dict_ib[basin], owz_res_dict_ib[basin] = compute_pod_and_far_by_basin(dynamicopy, 'owz', owz, ibtracs, max_track_distance_matching, basin, by_100=True)
for basin in basins:
    track_matches_dict_ib[basin], track_res_dict_ib[basin] = compute_pod_and_far_by_basin(dynamicopy, 'track', track, ibtracs, max_track_distance_matching, basin, by_100=True)
for basin in basins:
    uz_matches_dict_ib[basin], uz_res_dict_ib[basin] = compute_pod_and_far_by_basin(dynamicopy, 'uz', uz, ibtracs, max_track_distance_matching, basin, by_100=True)
for basin in basins:
    cnrm_matches_dict_ib[basin], cnrm_res_dict_ib[basin] = compute_pod_and_far_by_basin(dynamicopy, 'cnrm', cnrm, ibtracs, max_track_distance_matching, basin, by_100=True)
for basin in basins:
    bytestorm_matches_dict_ib[basin], bytestorm_res_dict_ib[basin] = compute_pod_and_far_by_basin(dynamicopy, our_work_name, bytestorm, ibtracs, max_track_distance_matching, basin, by_100=True)

In [ ]:
# Compute POD and FAR for each basin and for each TC Tracker
for basin in basins:
    owz_matches_dict_bo[basin], owz_res_dict_bo[basin] = compute_pod_and_far_by_basin(dynamicopy, 'owz', owz, ibtracs_bo, max_track_distance_matching, basin)
for basin in basins:
    track_matches_dict_bo[basin], track_res_dict_bo[basin] = compute_pod_and_far_by_basin(dynamicopy, 'track', track, ibtracs_bo, max_track_distance_matching, basin)
for basin in basins:
    uz_matches_dict_bo[basin], uz_res_dict_bo[basin] = compute_pod_and_far_by_basin(dynamicopy, 'uz', uz, ibtracs_bo, max_track_distance_matching, basin)
for basin in basins:
    cnrm_matches_dict_bo[basin], cnrm_res_dict_bo[basin] = compute_pod_and_far_by_basin(dynamicopy, 'cnrm', cnrm, ibtracs_bo, max_track_distance_matching, basin)
for basin in basins:
    bytestorm_matches_dict_bo[basin], bytestorm_res_dict_bo[basin] = compute_pod_and_far_by_basin(dynamicopy, our_work_name, bytestorm, ibtracs_bo, max_track_distance_matching, basin)

In [ ]:
# Determine the GridSpec Layout 
if len(basins) == 6:
    gs_rows, gs_cols = 2, 3
    gs_figsize=(20,8)
    gsix = lambda gs,i: gs[i//3, i % 3]
elif len(basins) == 7:
    gs_rows, gs_cols = 2, 8
    gs_figsize=(25,8)
    gsix = lambda gs,i: gs[0, i*2 : i*2+2] if i < 4 else gs[1, i%4*2+1 : i%4*2+3]

In [ ]:
fig = plt.figure(layout='constrained', figsize=gs_figsize)
gs = fig.add_gridspec(gs_rows, gs_cols)
for i,basin in enumerate([b for b in basins if b not in excluded_basins]):
    gsi = gsix(gs,i)
    axi = fig.add_subplot(gsi)
    algo_results_ib = pd.concat([owz_res_dict_ib[basin], track_res_dict_ib[basin], uz_res_dict_ib[basin], cnrm_res_dict_ib[basin], bytestorm_res_dict_ib[basin]]).reset_index(drop=True)
    plot_pod_and_far_on_axis(
        algo_results_ib,
        label=f' ({years[0]}-{years[-1]})',
        label_size=16,
        ticks_size=14,
        ax=axi,
        set_xlabel=True if i > 3 else False,
        outfile=None,
    )
    axi.text(-0.5, 100, basin, fontsize=20, weight='bold', color='black', ha='left', va='center')
    axi.spines[['right', 'top']].set_visible(False)
    if not i: handles, labels = axi.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', markerscale=12, edgecolor='gray', fontsize=16, framealpha=1, ncol=4, bbox_to_anchor=(0.5, 0))
plt.tight_layout()
outfile = os.path.join(out_dir, f'pod_far_by_basin_ibtracs.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
fig = plt.figure(layout='constrained', figsize=gs_figsize)
gs = fig.add_gridspec(gs_rows, gs_cols)
for i,basin in enumerate([b for b in basins if b not in excluded_basins]):
    gsi = gsix(gs,i)
    axi = fig.add_subplot(gsi)
    algo_results_bo = pd.concat([owz_res_dict_bo[basin], track_res_dict_bo[basin], uz_res_dict_bo[basin], cnrm_res_dict_bo[basin], bytestorm_res_dict_bo[basin]]).reset_index(drop=True)
    plot_pod_and_far_on_axis(
        algo_results_bo,
        label=f' ({years[0]}-{years[-1]})',
        label_size=16,
        ticks_size=14,
        ax=axi,
        set_xlabel=True if i > 3 else False,
        outfile=None,
    )
    axi.text(-0.5, 100, basin, fontsize=20, weight='bold', color='black', ha='left', va='center')
    axi.spines[['right', 'top']].set_visible(False)
    if not i: handles, labels = axi.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', markerscale=12, edgecolor='gray', fontsize=16, framealpha=1, ncol=4, bbox_to_anchor=(0.5, 0))
plt.tight_layout()
outfile = os.path.join(out_dir, f'pod_far_by_basin_bourdin.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

## POD and FAR by Storm Category

In [ ]:
ib = pd.read_csv(
    ibtracs_src, na_values=["", " "], usecols=[
        "SID","SEASON","BASIN","SUBBASIN","ISO_TIME","NATURE","TRACK_TYPE",
        "LON","LAT","WMO_WIND","USA_WIND","TOKYO_WIND","CMA_WIND","REUNION_WIND",
        "BOM_WIND","NADI_WIND","WELLINGTON_WIND","WMO_PRES","USA_PRES","TOKYO_PRES",
        "CMA_PRES","HKO_PRES","NEWDELHI_PRES","REUNION_PRES","BOM_PRES","NADI_PRES",
        "WELLINGTON_PRES",
    ],
    converters={
        "SID": str,"SEASON": int,"BASIN": str,"SUBBASIN": str,"LON": float,"LAT": float,
    },
    parse_dates=["ISO_TIME"],
)
# All about wind
## Select from the data & convert
ib["WIND10"] = np.where(
    ~ib.WMO_WIND.isna(),
    ib.WMO_WIND,
    ib[
        ["TOKYO_WIND", "REUNION_WIND", "BOM_WIND", "NADI_WIND", "WELLINGTON_WIND"]
    ].mean(axis=1, skipna=True),
)
ib["WIND10"] = np.where(
    ib.WIND10.isna(), ib.USA_WIND / 1.12, ib.WIND10
)  # Conversion rate in the IBTrACS doc
ib["WIND10"] = np.where(
    ib.WIND10.isna(), ib.CMA_WIND / 1.08, ib.WIND10
)  # Conversion rate determined through a linear regression
ib["WIND10"] *= 0.514  # Conversion noeuds en m/s

# Select pressure
ib["PRES"] = np.where(
    ~ib.WMO_PRES.isna(),
    ib.WMO_PRES,
    ib[
        [
            "USA_PRES","TOKYO_PRES","CMA_PRES","HKO_PRES","NEWDELHI_PRES",
            "REUNION_PRES","BOM_PRES","NADI_PRES","WELLINGTON_PRES",
        ]
    ].mean(axis=1, skipna=True),
)

# Rename columns
ib = ib.rename(columns={col: col.lower() for col in ib.columns}).rename(
    columns={
        "sid": "track_id",
        "pres": "slp",
        "iso_time": "time",
    }
)

# Geographical details
ib.loc[ib.lon < 0, "lon"] += 360
ib["hemisphere"] = np.where(ib.lat > 0, "N", "S")
ib["basin"] = dynamicopy.get_basin(ib.lon, ib.lat)

# All about time
ib["hour"] = ib.time.dt.hour
ib["day"] = ib.time.dt.day
ib["month"] = ib.time.dt.month
ib["year"] = ib.time.dt.year
ib = dynamicopy.add_season(ib)

## Filter tracks lasting for at least 4 time steps
tcs = (
    ib.groupby("track_id")["time"]
    .count()[ib.groupby("track_id")["time"].count() >= 4]
    .index
)
ib = ib[ib.track_id.isin(tcs)]

# Compute SSHS classification according to Klotzbach
ib["sshs"] = dynamicopy.sshs_from_pres(ib.slp)
ib["sshs_wind"] = dynamicopy.sshs_from_wind(ib.wind10)

ib = ib[
    [
        "track_id","time","lon","lat","hemisphere","basin","season",
        "sshs","sshs_wind","nature","slp","wind10","year","month","day","hour",
    ]
]

ib["ET"] = False

ib = ib[ib['time'].isin(dates)]

ib = convert_coords_to_format(ib, '[-180,180]')
ib

In [ ]:
matches_cat = pd.DataFrame()
res_cat = pd.DataFrame()
for cat in ib['sshs_wind'].unique():
    # cat_track_ids = ib[ib.sshs_wind==cat].track_id.unique()
    # ib_by_cat = ib[ib.track_id.isin(cat_track_ids)]
    ib_by_cat = ib[ib.sshs_wind==cat]

    owz_matches_cat, owz_cat_res = compute_pod_and_far(
        dynamicopy, owz, 'owz', ib_by_cat, max_track_distance_matching, print_results=False
    )
    track_matches_cat, track_cat_res = compute_pod_and_far(
        dynamicopy, track, 'track', ib_by_cat, max_track_distance_matching, print_results=False
    )
    uz_matches_cat, uz_cat_res = compute_pod_and_far(
        dynamicopy, uz, 'uz', ib_by_cat, max_track_distance_matching, print_results=False
    )
    cnrm_matches_cat, cnrm_cat_res = compute_pod_and_far(
        dynamicopy, cnrm, 'cnrm', ib_by_cat, max_track_distance_matching, print_results=False
    )
    bytestorm_matches_cat, bytestorm_cat_res = compute_pod_and_far(
        dynamicopy, bytestorm, our_work_name, ib_by_cat, max_track_distance_matching, print_results=False
    )

    # if cat == -1.0 or np.isnan(cat): cat_label = np.nan
    if cat == -1.0:
        cat_label = 'Depression'
    elif cat == 0.0:
        cat_label = 'Storm'
    elif np.isnan(cat):
        cat_label = np.nan
    else: cat_label = f'Cat {cat}'

    owz_matches_cat['CATEGORY'] = cat_label
    track_matches_cat['CATEGORY'] = cat_label
    uz_matches_cat['CATEGORY'] = cat_label
    cnrm_matches_cat['CATEGORY'] = cat_label
    bytestorm_matches_cat['CATEGORY'] = cat_label

    owz_cat_res['CATEGORY'] = cat_label
    track_cat_res['CATEGORY'] = cat_label
    uz_cat_res['CATEGORY'] = cat_label
    cnrm_cat_res['CATEGORY'] = cat_label
    bytestorm_cat_res['CATEGORY'] = cat_label

    owz_matches_cat['Tracker'] = 'OWZ'
    track_matches_cat['Tracker'] = 'TRACK'
    uz_matches_cat['Tracker'] = 'UZ'
    cnrm_matches_cat['Tracker'] = 'CNRM'
    bytestorm_matches_cat['Tracker'] = our_work_name

    matches_cat = pd.concat([
        matches_cat,
        owz_matches_cat,
        track_matches_cat,
        uz_matches_cat,
        cnrm_matches_cat,
        bytestorm_matches_cat,
    ])

    res_cat = pd.concat([
        res_cat,
        owz_cat_res,
        track_cat_res,
        uz_cat_res,
        cnrm_cat_res,
        bytestorm_cat_res,
    ])
matches_cat = matches_cat.reset_index(drop=True)
res_cat = res_cat.reset_index(drop=True)
res_cat['pod'] = res_cat['pod'] * 100
res_cat['far'] = res_cat['far'] * 100

In [ ]:
df = matches_cat.groupby(['CATEGORY','Tracker'], sort=False).mean(True).reset_index()[['CATEGORY','Tracker','matching_timesteps','dist']]

fig, ax = plt.subplots(1,1,figsize=(10,3))

sns.barplot(
    res_cat,
    x='algo',
    y='pod',
    hue='CATEGORY',
    hue_order=['Depression', 'Storm'] + [f'Cat {i}' for i in range(1,6)],
    saturation=1.0,
    palette=sns.color_palette("mako_r", n_colors=7),
    legend='auto',
    ax=ax,
)
ax.set_xlabel('TC Tracker')
ax.set_ylim([0,100])
ax.set_ylabel('POD')

box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
hs, ls = ax.get_legend_handles_labels()
ax.legend(
    hs, ls, loc='upper center', markerscale=10, edgecolor='gray', framealpha=1, 
    ncol=len(df.CATEGORY.unique()), bbox_to_anchor=(0.5 , -0.18))

outfile = os.path.join(out_dir, f"pod_by_storm_category.{file_ext}")
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
owz_overlap = dynamicopy.overlap(owz, ibtracs, owz_matches)
track_overlap = dynamicopy.overlap(track, ibtracs, track_matches)
uz_overlap = dynamicopy.overlap(uz, ibtracs, uz_matches)
cnrm_overlap = dynamicopy.overlap(cnrm, ibtracs, cnrm_matches)
bytestorm_overlap = dynamicopy.overlap(bytestorm, ibtracs, bytestorm_matches)

In [ ]:
plt.figure(figsize=(6,4))
owz_overlap['algo'] = INFO['owz']['label']
track_overlap['algo'] = INFO['track']['label']
uz_overlap['algo'] = INFO['uz']['label']
cnrm_overlap['algo'] = INFO['cnrm']['label']
bytestorm_overlap['algo'] = INFO['bytestorm']['label']
overlaps = pd.concat([
    owz_overlap,
    track_overlap,
    uz_overlap,
    cnrm_overlap,
    bytestorm_overlap,
])
sns.boxplot(overlaps, y='algo', x='delta_start', showfliers=False)
plt.xticks([-5, 0, 5, 10])
plt.vlines(x=0, ymin=-1, ymax=5, colors='black')
plt.show()

In [ ]:
bytestorm_matches_gen, bytestorm_res_gen = compute_pod_and_far(
    dynamicopy,
    bytestorm,
    our_work_name,
    # ib[ib.sshs_wind==cat]
    dynamicopy.genesis_points(ib[ib.sshs_wind==5]).reset_index(),
    max_track_distance_matching,
    print_results=False,
)
bytestorm_matches_gen

# Analysis over Storm NATUREs

In [ ]:
matches_nature = pd.DataFrame()
res_nature = pd.DataFrame()
for nature in ibtracs.NATURE.unique():
    owz_matches_nature, owz_nature_res = compute_pod_and_far(dynamicopy, owz, 'owz', ibtracs[ibtracs.NATURE==nature], max_track_distance_matching, print_results=False)
    track_matches_nature, track_nature_res = compute_pod_and_far(dynamicopy, track, 'track', ibtracs[ibtracs.NATURE==nature], max_track_distance_matching, print_results=False)
    uz_matches_nature, uz_nature_res = compute_pod_and_far(dynamicopy, uz, 'uz', ibtracs[ibtracs.NATURE==nature], max_track_distance_matching, print_results=False)
    cnrm_matches_nature, cnrm_nature_res = compute_pod_and_far(dynamicopy, cnrm, 'cnrm', ibtracs[ibtracs.NATURE==nature], max_track_distance_matching, print_results=False)
    bytestorm_matches_nature, bytestorm_nature_res = compute_pod_and_far(dynamicopy, bytestorm, our_work_name, ibtracs[ibtracs.NATURE==nature], max_track_distance_matching, print_results=False)

    owz_matches_nature['NATURE'] = nature
    track_matches_nature['NATURE'] = nature
    uz_matches_nature['NATURE'] = nature
    cnrm_matches_nature['NATURE'] = nature
    bytestorm_matches_nature['NATURE'] = nature

    owz_nature_res['NATURE'] = nature
    track_nature_res['NATURE'] = nature
    uz_nature_res['NATURE'] = nature
    cnrm_nature_res['NATURE'] = nature
    bytestorm_nature_res['NATURE'] = nature

    owz_matches_nature['Tracker'] = 'OWZ'
    track_matches_nature['Tracker'] = 'TRACK'
    uz_matches_nature['Tracker'] = 'UZ'
    cnrm_matches_nature['Tracker'] = 'CNRM'
    bytestorm_matches_nature['Tracker'] = our_work_name

    matches_nature = pd.concat([
        matches_nature,
        owz_matches_nature,
        track_matches_nature,
        uz_matches_nature,
        cnrm_matches_nature,
        bytestorm_matches_nature,
    ])

    res_nature = pd.concat([
        res_nature,
        owz_nature_res,
        track_nature_res,
        uz_nature_res,
        cnrm_nature_res,
        bytestorm_nature_res,
    ])
matches_nature = matches_nature.reset_index(drop=True)
res_nature = res_nature.reset_index(drop=True)
res_nature['pod'] = res_nature['pod'] * 100
res_nature['far'] = res_nature['far'] * 100

In [ ]:
df = matches_nature.groupby(['NATURE','Tracker'], sort=False).mean(True).reset_index()[['NATURE','Tracker','matching_timesteps','dist']]

fig, ax = plt.subplots(2,1,figsize=(13,8))

sns.barplot(res_nature, x='algo', y='pod', hue='NATURE', saturation=1.0, palette=sns.color_palette("deep"), legend=None, ax=ax[0])
ax[0].set_xlabel('TC Tracker')
ax[0].set_ylim([0,100])
ax[0].set_ylabel('POD [%]')
# plt.legend()

sns.barplot(df, x='Tracker', y='dist', hue='NATURE', saturation=1.0, palette=sns.color_palette("deep"), ax=ax[1])
ax[1].set_xlabel('TC Tracker')
ax[1].set_ylabel('Distance Error [km]')
# plt.legend()

box = ax[1].get_position()
ax[1].set_position([box.x0, box.y0, box.width, box.height])
ax[1].legend(loc='upper center', markerscale=10, edgecolor='gray', framealpha=1, ncol=len(ibtracs.NATURE.unique()), bbox_to_anchor=(0.5 , -0.14))

outfile = os.path.join(out_dir, f"tc_dist_and_pod_by_nature.{file_ext}")
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

## Enriched POD and FAR: Success Ratio, Critical Success Index and Bias

In [ ]:
algo_results['sr'] = algo_results['sr'] * 100
algo_results['csi'] = algo_results['csi'] * 100
algo_results['bias'] = algo_results['bias']

In [ ]:
label = '' # f' {years[0]}-{years[-1]}'
x = np.arange(len(algo_results))  # the label locations
width = 0.2  # the width of the bars
multiplier = 0

fig, ax = plt.subplots(layout='constrained', figsize=(12,6))

offset = width * multiplier
rects = ax.bar(x + offset, np.round(algo_results['pod'],2), width, label=f'POD' + label, color='tab:blue')
ax.bar_label(rects, padding=3)
multiplier += 1

offset = width * multiplier
rects = ax.bar(x + offset, np.round(algo_results['far'],2), width, label=f'FAR' + label, color='tab:red')
ax.bar_label(rects, padding=3)
multiplier += 1

offset = width * multiplier
rects = ax.bar(x + offset, np.round(algo_results['csi'],2), width, label=f'CSI' + label, color='tab:orange')
ax.bar_label(rects, padding=3)
multiplier += 1

offset = width * multiplier
rects = ax.bar(x + offset, np.round(algo_results['sr'],2), width, label=f'SR' + label, color='tab:green')
ax.bar_label(rects, padding=3)
multiplier += 1

# Add some text for labels, title and custom x-axis tick labels, etc.
ax.set_ylabel('Score', size = 14)
ax.set_xlabel('TC trackers', size = 14)
ax.set_xticks(x + width*3/2, algo_results['algo'], fontsize=12)
ax.set_ylim(0, 100)

box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
ax.legend(loc='center', markerscale=10, edgecolor='gray', framealpha=1, ncol=4, bbox_to_anchor=(0.5 , -0.15))

plt.tight_layout()
outfile = os.path.join(out_dir, f'pod_far_csi_sr.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
outfile = os.path.join(out_dir, 'tableI.csv')
table_I = pd.DataFrame()
for basin in basins:
    owz_res_dict_ib_basin = owz_res_dict_ib[basin]
    owz_res_dict_ib_basin['basin'] = basin
    track_res_dict_ib_basin = track_res_dict_ib[basin]
    track_res_dict_ib_basin['basin'] = basin
    uz_res_dict_ib_basin = uz_res_dict_ib[basin]
    uz_res_dict_ib_basin['basin'] = basin
    cnrm_res_dict_ib_basin = cnrm_res_dict_ib[basin]
    cnrm_res_dict_ib_basin['basin'] = basin
    bytestorm_res_dict_ib_basin = bytestorm_res_dict_ib[basin]
    bytestorm_res_dict_ib_basin['basin'] = basin
    table_I = pd.concat([
        table_I,
        owz_res_dict_ib_basin,
        track_res_dict_ib_basin,
        uz_res_dict_ib_basin,
        cnrm_res_dict_ib_basin,
        bytestorm_res_dict_ib_basin,
    ])
table_I['pod'] = np.around(table_I['pod'],2)
table_I['far'] = np.around(table_I['far'],2)
table_I['csi'] = np.around(table_I['csi'],2)
table_I['sr'] = np.around(table_I['sr'],2)
table_I['bias'] = np.around(table_I['bias'],2)
table_I.reset_index(drop=True).to_csv(outfile) if save_figures else None

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def plot_advanced_performance_diagram(
        basins: list,
        owz_res_dict: dict,
        track_res_dict: dict,
        uz_res_dict: dict,
        cnrm_res_dict: dict,
        bytestorm_res_dict: dict,
        divide_by_100: bool = False,
        outfile: str = None,
    ):
    plt.figure(figsize=(6, 5))
    
    # --- Background: Isolinee CSI e Linee Bias ---
    x = np.linspace(0.01, 1, 100)
    y = np.linspace(0.01, 1, 100)
    sr_grid, pod_grid = np.meshgrid(x, y)
    csi_grid = 1 / (1/sr_grid + 1/pod_grid - 1)

    csi_contour = plt.contour(sr_grid, pod_grid, csi_grid, levels=np.arange(0.1, 1.0, 0.1), 
                              colors='lightgrey', linestyles='--', linewidths=0.8)
    plt.clabel(csi_contour, inline=True, fontsize=9, fmt='%.1f')

    for b in [0.5, 0.8, 1.0, 1.25, 2.0]:
        plt.plot(x, b * x, color='gray', alpha=0.2, linewidth=1)
        plt.text(0.95 if b <= 1 else 0.95/b, 0.95*b if b <= 1 else 0.95, f'B={b}', 
                 fontsize=8, alpha=0.4, ha='center')

    # --- Mappatura Simboli (Modelli) e Colori (Bacini) ---
    model_markers = {
        INFO['owz']['label']: 's',   # Quadrato
        INFO['track']['label']: '^', # Triangolo
        INFO['uz']['label']: 'D',    # Diamante
        INFO['cnrm']['label']: 'v',  # Triangolo giù
        our_work_name: '*',          # Cerchio
    }

    basin_colors = {
        'WNP': 'tab:blue',
        'ENP': 'tab:orange',
        'NI': 'tab:green',
        'SI': 'tab:red',
        'SP': 'tab:purple',
        'NATL': 'tab:brown',
    }

    for basin in basins:
        owz_res_basin = owz_res_dict[basin].copy()
        track_res_basin = track_res_dict[basin].copy()
        uz_res_basin = uz_res_dict[basin].copy()
        cnrm_res_basin = cnrm_res_dict[basin].copy()
        bytestorm_res_basin = bytestorm_res_dict[basin].copy()
        x, y = owz_res_basin['sr'], owz_res_basin['pod']
        if divide_by_100:
            x /= 100
            y /= 100
        plt.scatter(x, y, marker=model_markers[INFO['owz']['label']], color=basin_colors[basin], alpha=0.6)
        x, y = track_res_basin['sr'], track_res_basin['pod']
        if divide_by_100:
            x /= 100
            y /= 100
        plt.scatter(x, y, marker=model_markers[INFO['track']['label']], color=basin_colors[basin], alpha=0.6)
        x, y = uz_res_basin['sr'], uz_res_basin['pod']
        if divide_by_100:
            x /= 100
            y /= 100
        plt.scatter(x, y, marker=model_markers[INFO['uz']['label']], color=basin_colors[basin], alpha=0.6)
        x, y = cnrm_res_basin['sr'], cnrm_res_basin['pod']
        if divide_by_100:
            x /= 100
            y /= 100
        plt.scatter(x, y, marker=model_markers[INFO['cnrm']['label']], color=basin_colors[basin], alpha=0.6)
        x, y = bytestorm_res_basin['sr'], bytestorm_res_basin['pod']
        if divide_by_100:
            x /= 100
            y /= 100
        plt.scatter(x, y, marker=model_markers[our_work_name], color=basin_colors[basin], s=100)

    # --- Legende Custom ---
    from matplotlib.lines import Line2D
    
    # Legenda 1: Modelli (Nero, diversi simboli)
    model_legend = [Line2D([0], [0], marker=m, color='w', markerfacecolor='gray', 
                           markersize=10, label=mod, markeredgecolor='k') 
                    for mod, m in model_markers.items()]
    
    # Legenda 2: Bacini (Cerchi, diversi colori)
    basin_legend = [Line2D([0], [0], marker='o', color='w', markerfacecolor=c, 
                           markersize=10, label=bas) 
                    for bas, c in basin_colors.items()]

    l1 = plt.legend(handles=basin_legend, title="Basins", loc='upper left', bbox_to_anchor=(1.0, 0.5))
    plt.gca().add_artist(l1)
    l2 = plt.legend(handles=model_legend, title="Models", loc='upper left', bbox_to_anchor=(1.0, 1))

    plt.xlabel('Success Ratio (1 - FAR)', fontsize=12)
    plt.ylabel('Probability of Detection (POD)', fontsize=12)
    plt.title('Roebber Diagram', fontsize=14, pad=20)
    plt.xlim(0, 1); plt.ylim(0, 1)
    plt.grid(alpha=0.1)
    # plt.tight_layout()
    # plt.subplots_adjust(right=0.5)
    plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

outfile = os.path.join(out_dir, f'roebber_diagram_by_basins.{file_ext}')
plot_advanced_performance_diagram(
    basins,
    owz_res_dict_ib,
    track_res_dict_ib,
    uz_res_dict_ib,
    cnrm_res_dict_ib,
    bytestorm_res_dict_ib,
    divide_by_100=True,
    outfile=outfile if save_figures else None,
)

# Track Smoothness

In [ ]:
smoothness = pd.DataFrame(
    columns = ['algo', 'score', 'delta_angles']
)

for track_id in ibtracs.track_id.unique():
    latlon = ibtracs[ibtracs['track_id']==track_id][['lat','lon']].to_numpy()
    lat, lon = latlon[:,0], latlon[:,1]
    score, delta_angles = track_smoothness(lat, lon)
    smoothness = pd.concat([
        smoothness, 
        pd.DataFrame({
            'algo': ['ibtracs'], 
            'track_id': [track_id], 
            'score': [score], 
            'delta_angles': [delta_angles], 
        })
    ])

for algo, matches, tracker, idname in zip(
        ['owz', 'track', 'uz', 'cnrm', f'{our_work_name}'], 
        [owz_matches, track_matches, uz_matches, cnrm_matches, bytestorm_matches], 
        [owz, track, uz, cnrm, bytestorm], 
        ['id_owz', 'id_track', 'id_uz', 'id_cnrm', f'id_{our_work_name}']
    ):
    for track_id in tracker.track_id.unique():
    # for track_id in tracker.track_id.unique():
        latlon = tracker[tracker['track_id']==track_id][['lat','lon']].to_numpy()
        lat, lon = latlon[:,0], latlon[:,1]
        score, delta_angles = track_smoothness(lat, lon)
        smoothness = pd.concat([
            smoothness, 
            pd.DataFrame({
                'algo': [algo], 
                'track_id': [track_id], 
                'score': [score], 
                'delta_angles': [delta_angles], 
            })
        ])
smoothness = smoothness.reset_index(drop=True)
smoothness

In [ ]:
palette = [
    INFO['ibtracs']['color'], INFO['owz']['color'], INFO['track']['color'], INFO['uz']['color'], INFO['cnrm']['color'], INFO['bytestorm']['color']
]
fig, ax = plt.subplots(1, 1, figsize=(13,6))

ax = sns.boxenplot(smoothness, x='algo', y='score', palette=palette)
ax.set_xlabel("TC Trackers", fontdict={'size':14})
ax.set_ylabel("Track Smoothness", fontdict={'size':14})
ax.set_xticklabels(["IBTrACS", "OWZ", "TRACK", "UZ", "CNRM", our_work_name.upper()])
ax.set_title("Track Smoothness for TC Tracks\non the Test Set", fontdict={'size':14})

outfile = os.path.join(out_dir, f'track_smoothness_boxplot.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

## POD and FAR after removing noisy TC Tracks

In [ ]:
q = 0.9

bytestorm_smooth = pd.merge(bytestorm, smoothness[['score','track_id']], on='track_id')
bytestorm_quantile_075 = bytestorm_smooth['score'].quantile(q=q)
bytestorm_smooth = bytestorm_smooth[bytestorm_smooth['score'] <= bytestorm_quantile_075]

# compute the POD and FAR
bytestorm_matches_smooth, bytestorm_res_smooth = compute_pod_and_far(dynamicopy, bytestorm_smooth, our_work_name, ibtracs, max_track_distance_matching, print_results=False)

algo_results_smooth = pd.concat([owz_res, track_res, uz_res, cnrm_res, bytestorm_res_smooth]).reset_index(drop=True)
algo_results_smooth['model'] = selected_model
algo_results_smooth['pod'] = algo_results_smooth['pod'] * 100
algo_results_smooth['far'] = algo_results_smooth['far'] * 100

fig, ax = plt.subplots(1, 1, layout='constrained', figsize=(12,6))
plot_pod_and_far_on_axis(algo_results_smooth, label=f' ({years[0]}-{years[-1]})', ax=ax, set_xlabel=True, outfile=None)

handles, labels = ax.get_legend_handles_labels()
fig.legend(handles, labels, loc='upper center', markerscale=12, edgecolor='gray', framealpha=1, ncol=4, bbox_to_anchor=(0.5, 0))

plt.tight_layout()
outfile = os.path.join(out_dir, f'pod_far_smoother_tcs.{file_ext}') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

## Which TCs have noisier TC Tracks?

In [ ]:
# get the tracks that are removed after filtering over noisier TCs
merge_bytestorm = pd.merge(bytestorm_matches_smooth, bytestorm_matches_bo, on=[f'id_{our_work_name}', 'id_ibtracs'], how='right')
# remove TCs where we have NaN (they have been removed after filtering)
merge_bytestorm = merge_bytestorm[merge_bytestorm['dist_x'].isna()]
# merge the DataFrame with ibtracs to get more info about these TCs
merge_bytestorm = pd.merge(merge_bytestorm, ibtracs, left_on='id_ibtracs', right_on='track_id')

plt.figure(figsize=(8,4))
sns.barplot(merge_bytestorm.groupby(by='NATURE')['NATURE'].value_counts(), palette=sns.color_palette('deep'))
plt.xlabel('Storm Nature', fontdict=dict(size=12))
plt.ylabel('# TC Detections', fontdict=dict(size=12))
outfile = os.path.join(out_dir, f'noisy_tcs_by_nature.{file_ext}') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
merge_bytestorm = pd.merge(merge_bytestorm, ibtracs_bo, left_on=['id_ibtracs', 'time'], right_on=['track_id', 'time'])
wind_by_nature_df = pd.DataFrame(
    data=dict(
        WIND=[float(w) for w,n in zip(merge_bytestorm['wind10'], merge_bytestorm['NATURE']) if w != ' '],
        NATURE=[n for w,n in zip(merge_bytestorm['wind10'], merge_bytestorm['NATURE']) if w != ' '],
    ),
)

In [ ]:
winds = np.array([float(w) for w in merge_bytestorm['wind10'] if w != ' '])
plt.figure(figsize=(8,4))
plt.hist(winds, color='tab:blue')
plt.xlabel('MSW [m/s]', fontdict=dict(size=12))
plt.ylabel('# TC Detections', fontdict=dict(size=12))
outfile = os.path.join(out_dir, f'noisy_tcs_by_msw.{file_ext}') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

# LAT-LAT and LON-LON Plots

In [ ]:
merges = []
for i, row in bytestorm_matches.iterrows():
    bytestorm_id, ibtracs_id = row[f'id_{our_work_name}'], row['id_ibtracs']
    merge = pd.merge(bytestorm[bytestorm['track_id']==bytestorm_id], ibtracs[ibtracs['track_id']==ibtracs_id], on='time')
    merge['dist'] = haversine_vector(merge[['lat_x','lon_x']].to_numpy(), merge[['lat_y','lon_y']].to_numpy())
    merges.append(merge)
merge = pd.concat(merges).reset_index(drop=True)
merge.loc[merge['WMO_WIND']==' ', 'WMO_WIND'] = merge[merge['WMO_WIND']!=' ']['WMO_WIND'].astype(float).mean()
merge.head()

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(8,4))

fig.suptitle('True/Predicted LAT and LON Distributions\nover Test Set', size=11, weight='bold')

im = ax[0].scatter(merge['lat_x'], merge['lat_y'], marker='.', c=merge['WMO_WIND'].astype(float), cmap='cool')
ax[0].plot([merge['lat_x'].min(), merge['lat_x'].max()], [merge['lat_y'].min(), merge['lat_y'].max()], color='r', linewidth=1)
ax[0].set_xlabel('Pred Latitude', fontdict={'size':10, 'weight': 'bold'})
ax[0].set_ylabel('True Latitude', fontdict={'size':10, 'weight': 'bold'})

merge['lon_x'] = merge['lon_x'] % 360
merge['lon_y'] = merge['lon_y'] % 360

im = ax[1].scatter(merge['lon_x'], merge['lon_y'], marker='.', c=merge['WMO_WIND'].astype(float), cmap='cool')
ax[1].plot([merge['lon_x'].min(), merge['lon_x'].max()], [merge['lon_y'].min(), merge['lon_y'].max()], color='r', linewidth=1)
ax[1].set_xlabel('Pred Longitude', fontdict={'size':10, 'weight': 'bold'})
ax[1].set_ylabel('True Longitude', fontdict={'size':10, 'weight': 'bold'})
cb = fig.colorbar(im, ax=ax[1])
cb.set_label(label='MSW [kn]', size=10, weight='bold')

plt.tight_layout()
outfile = os.path.join(out_dir, f'lat_lon_distribution.png') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

# Top 5 Longest Tracks Comparison

In [ ]:
longest_ibtracs_track_ids = ibtracs.value_counts('track_id')[:5].reset_index()['track_id'].to_numpy()
ibtracs_longest_tracks = ibtracs[ibtracs['track_id'].isin(longest_ibtracs_track_ids)]
longest_ibtracs_track_ids

In [ ]:
ibtracs_longest_tracks.NATURE.unique()

In [ ]:
# get only TS storms
ibtracs_longest_tracks = ibtracs_longest_tracks[ibtracs_longest_tracks['NATURE'].isin(['TS', 'ET'])]

In [ ]:
owz_matches_lts, owz_res_lts = compute_pod_and_far(dynamicopy, owz, 'owz', ibtracs_longest_tracks, max_track_distance_matching, print_results=False)
track_matches_lts, track_res_lts = compute_pod_and_far(dynamicopy, track, 'track', ibtracs_longest_tracks, max_track_distance_matching, print_results=False)
uz_matches_lts, uz_res_lts = compute_pod_and_far(dynamicopy, uz, 'uz', ibtracs_longest_tracks, max_track_distance_matching, print_results=False)
cnrm_matches_lts, cnrm_res_lts = compute_pod_and_far(dynamicopy, cnrm, 'cnrm', ibtracs_longest_tracks, max_track_distance_matching, print_results=False)
bytestorm_matches_lts, bytestorm_res_lts = compute_pod_and_far(dynamicopy, bytestorm, our_work_name, ibtracs_longest_tracks, max_track_distance_matching, print_results=False)

In [ ]:
owz_matches_lts = pd.merge(owz_matches_lts, ibtracs_longest_tracks.rename(columns={'track_id': 'id_ibtracs'}).value_counts('id_ibtracs').reset_index(), on='id_ibtracs')
track_matches_lts = pd.merge(track_matches_lts, ibtracs_longest_tracks.rename(columns={'track_id': 'id_ibtracs'}).value_counts('id_ibtracs').reset_index(), on='id_ibtracs')
uz_matches_lts = pd.merge(uz_matches_lts, ibtracs_longest_tracks.rename(columns={'track_id': 'id_ibtracs'}).value_counts('id_ibtracs').reset_index(), on='id_ibtracs')
cnrm_matches_lts = pd.merge(cnrm_matches_lts, ibtracs_longest_tracks.rename(columns={'track_id': 'id_ibtracs'}).value_counts('id_ibtracs').reset_index(), on='id_ibtracs')
bytestorm_matches_lts = pd.merge(bytestorm_matches_lts, ibtracs_longest_tracks.rename(columns={'track_id': 'id_ibtracs'}).value_counts('id_ibtracs').reset_index(), on='id_ibtracs')

In [ ]:
# load original ibtracs
ori_ibtracs = pd.read_csv(ibtracs_src, index_col=0).rename(columns={'ISO_TIME':'time','LAT':'lat','LON':'lon','SID':'track_id'})[['time','lat','lon','track_id','NATURE','WMO_WIND','NAME']]

In [ ]:
# find the track ids and the corresponding name
track_id_and_name = pd.DataFrame({
    'track_id': longest_ibtracs_track_ids, 
    'name': [ori_ibtracs[ori_ibtracs['track_id']==tid].iloc[0]['NAME'] for tid in longest_ibtracs_track_ids] 
})
track_id_and_name

In [ ]:
owz_matches_lts = pd.merge(owz_matches_lts, track_id_and_name.rename(columns={'track_id':'id_ibtracs'}), on='id_ibtracs')
track_matches_lts = pd.merge(track_matches_lts, track_id_and_name.rename(columns={'track_id':'id_ibtracs'}), on='id_ibtracs')
uz_matches_lts = pd.merge(uz_matches_lts, track_id_and_name.rename(columns={'track_id':'id_ibtracs'}), on='id_ibtracs')
cnrm_matches_lts = pd.merge(cnrm_matches_lts, track_id_and_name.rename(columns={'track_id':'id_ibtracs'}), on='id_ibtracs')
bytestorm_matches_lts = pd.merge(bytestorm_matches_lts, track_id_and_name.rename(columns={'track_id':'id_ibtracs'}), on='id_ibtracs')

In [ ]:
owz_matches_lts['perc'] = (owz_matches_lts['matching_timesteps'] / owz_matches_lts['count']) * 100
track_matches_lts['perc'] = (track_matches_lts['matching_timesteps'] / track_matches_lts['count']) * 100
uz_matches_lts['perc'] = (uz_matches_lts['matching_timesteps'] / uz_matches_lts['count']) * 100
cnrm_matches_lts['perc'] = (cnrm_matches_lts['matching_timesteps'] / cnrm_matches_lts['count']) * 100
bytestorm_matches_lts['perc'] = (bytestorm_matches_lts['matching_timesteps'] / bytestorm_matches_lts['count']) * 100

# drop duplicates
owz_matches_lts = owz_matches_lts.drop_duplicates('id_ibtracs')
track_matches_lts = track_matches_lts.drop_duplicates('id_ibtracs')
uz_matches_lts = uz_matches_lts.drop_duplicates('id_ibtracs')
cnrm_matches_lts = cnrm_matches_lts.drop_duplicates('id_ibtracs')
bytestorm_matches_lts = bytestorm_matches_lts.drop_duplicates('id_ibtracs')

In [ ]:
not_found_tc_names = [name for name in track_id_and_name['name'].values if name not in owz_matches_lts['name'].values]
if len(not_found_tc_names) != 0:
    owz_matches_lts = pd.concat([owz_matches_lts, pd.DataFrame(data={
        f'id_{our_work_name}':[None] * len(not_found_tc_names),
        'id_ibtracs':[None] * len(not_found_tc_names),
        'matching_timesteps':[0.0] * len(not_found_tc_names),
        'dist':[0.0] * len(not_found_tc_names),
        'count':[0.0] * len(not_found_tc_names),
        'name': not_found_tc_names,
        'perc':[0] * len(not_found_tc_names),
    })])

not_found_tc_names = [name for name in track_id_and_name['name'].values if name not in track_matches_lts['name'].values]
if len(not_found_tc_names) != 0:
    track_matches_lts = pd.concat([track_matches_lts, pd.DataFrame(data={
        f'id_{our_work_name}':[None] * len(not_found_tc_names),
        'id_ibtracs':[None] * len(not_found_tc_names),
        'matching_timesteps':[0.0] * len(not_found_tc_names),
        'dist':[0.0] * len(not_found_tc_names),
        'count':[0.0] * len(not_found_tc_names),
        'name': not_found_tc_names,
        'perc':[0] * len(not_found_tc_names),
    })])

not_found_tc_names = [name for name in track_id_and_name['name'].values if name not in uz_matches_lts['name'].values]
if len(not_found_tc_names) != 0:
    uz_matches_lts = pd.concat([uz_matches_lts, pd.DataFrame(data={
        f'id_{our_work_name}':[None] * len(not_found_tc_names),
        'id_ibtracs':[None] * len(not_found_tc_names),
        'matching_timesteps':[0.0] * len(not_found_tc_names),
        'dist':[0.0] * len(not_found_tc_names),
        'count':[0.0] * len(not_found_tc_names),
        'name': not_found_tc_names,
        'perc':[0] * len(not_found_tc_names),
    })])

not_found_tc_names = [name for name in track_id_and_name['name'].values if name not in cnrm_matches_lts['name'].values]
if len(not_found_tc_names) != 0:
    cnrm_matches_lts = pd.concat([cnrm_matches_lts, pd.DataFrame(data={
        f'id_{our_work_name}':[None] * len(not_found_tc_names),
        'id_ibtracs':[None] * len(not_found_tc_names),
        'matching_timesteps':[0.0] * len(not_found_tc_names),
        'dist':[0.0] * len(not_found_tc_names),
        'count':[0.0] * len(not_found_tc_names),
        'name': not_found_tc_names,
        'perc':[0] * len(not_found_tc_names),
    })])

not_found_tc_names = [name for name in track_id_and_name['name'].values if name not in bytestorm_matches_lts['name'].values]
if len(not_found_tc_names) != 0:
    bytestorm_matches_lts = pd.concat([bytestorm_matches_lts, pd.DataFrame(data={
        f'id_{our_work_name}':[None] * len(not_found_tc_names),
        'id_ibtracs':[None] * len(not_found_tc_names),
        'matching_timesteps':[0.0] * len(not_found_tc_names),
        'dist':[0.0] * len(not_found_tc_names),
        'count':[0.0] * len(not_found_tc_names),
        'name': not_found_tc_names,
        'perc':[0] * len(not_found_tc_names),
    })])

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(14, 6))
x = np.arange(len(longest_ibtracs_track_ids))
width = 0.1
mult = 0

offset = width * mult
rects = ax.bar(x + offset, [np.around(val, 0) for val in owz_matches_lts['perc'].to_numpy()], width, label=INFO['owz']['label'], color=INFO['owz']['color'])
ax.bar_label(rects, padding=3)
mult += 1

offset = width * mult
rects = ax.bar(x + offset, [np.around(val, 0) for val in track_matches_lts['perc'].to_numpy()], width, label=INFO['track']['label'], color=INFO['track']['color'])
ax.bar_label(rects, padding=3)
mult += 1

offset = width * mult
rects = ax.bar(x + offset, [np.around(val, 0) for val in uz_matches_lts['perc'].to_numpy()], width, label=INFO['uz']['label'], color=INFO['uz']['color'])
ax.bar_label(rects, padding=3)
mult += 1

offset = width * mult
rects = ax.bar(x + offset, [np.around(val, 0) for val in cnrm_matches_lts['perc'].to_numpy()], width, label=INFO['cnrm']['label'], color=INFO['cnrm']['color'])
ax.bar_label(rects, padding=3)
mult += 1

offset = width * mult
rects = ax.bar(x + offset, [np.around(val, 0) for val in bytestorm_matches_lts['perc'].to_numpy()], width, label=INFO['bytestorm']['label'], color=INFO['bytestorm']['color'])
ax.bar_label(rects, padding=3)
mult += 1


ax.set_xticks(x + width * 2)
ax.set_xticklabels(
    [track_id_and_name[track_id_and_name['track_id']==track_id]['name'].to_numpy()[0] for track_id in longest_ibtracs_track_ids], 
    fontdict = {
        # 'weight': 'bold', 
        'size': 12, 
    }
)
ax.set_ylabel(
    'Track Matched [%]', 
    size = 14, 
    # weight = 'bold', 
)
ax.set_ylim(0, 110)

# add the legend
# plt.legend()
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
ax.legend(loc='upper left', markerscale=10, edgecolor='gray', framealpha=1, ncol=5, bbox_to_anchor=(0.23, -0.08))

plt.title(
    'Percentage of Track Matched on the 5 Longest TC Tracks of the Test Set', 
    fontdict = {
        # 'weight': 'bold', 
        'size':16, 
    }
)
plt.tight_layout()

outfile = os.path.join(out_dir, f'top_5_longest_tracks_bar.{file_ext}') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
bytestorm_lts_track_id = bytestorm_matches_lts[bytestorm_matches_lts['id_ibtracs']==longest_ibtracs_track_ids[0]][f'id_{our_work_name}'].iloc[0]
total_number_of_detections = len(bytestorm[bytestorm['track_id']==bytestorm_lts_track_id])
number_of_matching_timesteps = int(bytestorm_matches_lts[bytestorm_matches_lts['id_ibtracs']==longest_ibtracs_track_ids[0]]['matching_timesteps'].iloc[0])
total_number_of_observations = int(bytestorm_matches_lts[bytestorm_matches_lts['id_ibtracs']==longest_ibtracs_track_ids[0]]['count'].iloc[0])
number_of_matching_timesteps, total_number_of_observations, total_number_of_detections

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(12, 5))
x = np.arange(len(longest_ibtracs_track_ids))
width = 0.1
mult = 0

offset = width * mult
rects = ax.bar(x + offset, [np.around(val, 0) for val in owz_matches_lts['dist'].to_numpy()], width, label=INFO['owz']['label'], color=INFO['owz']['color'])
ax.bar_label(rects, padding=3)
mult += 1

offset = width * mult
rects = ax.bar(x + offset, [np.around(val, 0) for val in track_matches_lts['dist'].to_numpy()], width, label=INFO['track']['label'], color=INFO['track']['color'])
ax.bar_label(rects, padding=3)
mult += 1

offset = width * mult
rects = ax.bar(x + offset, [np.around(val, 0) for val in uz_matches_lts['dist'].to_numpy()], width, label=INFO['uz']['label'], color=INFO['uz']['color'])
ax.bar_label(rects, padding=3)
mult += 1

offset = width * mult
rects = ax.bar(x + offset, [np.around(val, 0) for val in cnrm_matches_lts['dist'].to_numpy()], width, label=INFO['cnrm']['label'], color=INFO['cnrm']['color'])
ax.bar_label(rects, padding=3)
mult += 1

offset = width * mult
rects = ax.bar(x + offset, [np.around(val, 0) for val in bytestorm_matches_lts['dist'].to_numpy()], width, label=INFO['bytestorm']['label'], color=INFO['bytestorm']['color'])
ax.bar_label(rects, padding=3)
mult += 1

ax.set_xticks(x + width * 2)
ax.set_xticklabels(
    [track_id_and_name[track_id_and_name['track_id']==track_id]['name'].to_numpy()[0] for track_id in longest_ibtracs_track_ids], 
    fontdict={'weight': 'bold', 'size': 12}
)
ax.set_ylabel(
    'Distance [km]', 
    size = 14, 
    weight = 'bold')
# ax.set_ylim(0, 105)

# add the legend
# plt.legend()
box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width, box.height])
ax.legend(loc='upper left', markerscale=10, edgecolor='gray', framealpha=1, ncol=5, bbox_to_anchor=(0.23, -0.08))

plt.tight_layout()

plt.show()

In [ ]:
owz_longest_tracks = owz[owz['track_id'].isin(owz_matches_lts['id_owz'].unique())]
track_longest_tracks = track[track['track_id'].isin(track_matches_lts['id_track'].unique())]
uz_longest_tracks = uz[uz['track_id'].isin(uz_matches_lts['id_uz'].unique())]
cnrm_longest_tracks = cnrm[cnrm['track_id'].isin(cnrm_matches_lts['id_cnrm'].unique())]
bytestorm_longest_tracks = bytestorm[bytestorm['track_id'].isin(bytestorm_matches_lts['id_'+our_work_name].unique())]

ibtracs_longest_tracks['lon'] = ibtracs_longest_tracks['lon'] % 360
owz_longest_tracks['lon'] = owz_longest_tracks['lon'] % 360
track_longest_tracks['lon'] = track_longest_tracks['lon'] % 360
uz_longest_tracks['lon'] = uz_longest_tracks['lon'] % 360
cnrm_longest_tracks['lon'] = cnrm_longest_tracks['lon'] % 360
bytestorm_longest_tracks['lon'] = bytestorm_longest_tracks['lon'] % 360

title = 'Comparison'

# set map extent
central_longitude = (lon_range[1] - lon_range[0]) // 2

fig = plt.figure(figsize=(25,10))
proj = ccrs.PlateCarree(central_longitude=central_longitude)
ax = plt.axes(projection=proj)

image_extent = [lon_range[0], lon_range[1], lat_range[0], lat_range[1]]
ax.set_extent(image_extent, crs=ccrs.PlateCarree())
ax.coastlines(resolution='50m', lw=0.2)
ax.add_feature(cf.LAND, facecolor='lightgrey', alpha=0.3)

fontdict = {'weight':'bold', 'size':14}
title_fontdict = {'size':18}
ticksize = 12
track_id_col = 'TRACK_ID' if 'TRACK_ID' in ibtracs_longest_tracks.columns else 'track_id'
lon_col = 'LON' if 'LON' in ibtracs_longest_tracks.columns else 'lon'
lat_col = 'LAT' if 'LAT' in ibtracs_longest_tracks.columns else 'lat'

# plot tracks in each basin
alpha = 0.5
marker_size = 10.0
transform = ccrs.PlateCarree()
plt.title(title, fontsize = 18)

for tracker in [
        ibtracs_longest_tracks, 
        owz_longest_tracks, 
        track_longest_tracks, 
        uz_longest_tracks, 
        cnrm_longest_tracks, 
        bytestorm_longest_tracks, 
    ]:
    for i, id in enumerate(tracker[track_id_col].unique()):
        ax.plot(tracker[tracker[track_id_col]==id][lon_col], tracker[tracker[track_id_col]==id][lat_col], alpha=0.2, transform=transform)
        ax.scatter(tracker[tracker[track_id_col]==id][lon_col], tracker[tracker[track_id_col]==id][lat_col], s=marker_size, marker='o', alpha=0.9, transform=transform, label=f'' if i==0 else None)

# x-axis
longitudes = np.arange(lon_range[0], lon_range[1]+1, 10)
lon_formatter = LongitudeFormatter(zero_direction_label=False)
ax.xaxis.set_major_formatter(lon_formatter)
ax.xaxis.set_major_locator(mticker.FixedLocator(longitudes-central_longitude))
ax.set_xticklabels(longitudes, size=ticksize)
ax.set_xticks(longitudes-central_longitude)
ax.set_xlabel('Longitude [deg]', fontdict=fontdict)

# y-axis
latitudes = np.arange(lat_range[0], lat_range[1]+1, 10)
lat_formatter = LatitudeFormatter()
ax.yaxis.set_major_formatter(lat_formatter)
ax.yaxis.set_major_locator(mticker.FixedLocator(latitudes))
ax.set_yticklabels(latitudes, size=ticksize)
ax.set_yticks(latitudes)
ax.set_ylabel('Latitude [deg]', fontdict=fontdict)

# gridlines
gl = ax.gridlines(crs=proj, draw_labels=False, dms=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.xlocator = mticker.FixedLocator(longitudes-central_longitude)
gl.ylocator = mticker.FixedLocator(latitudes)
gl.xlines = True
gl.ylines = True

box = ax.get_position()
ax.set_position([box.x0, box.y0, box.width * 0.8, box.height])
ax.legend(loc='upper left', markerscale=2., edgecolor='black', framealpha=1, ncol=4, fontsize=14, bbox_to_anchor=(0.29,-0.11))
plt.show()

# Spatial Distribution

Plot the spatial distribution of TC detections over latitude and longitude

In [ ]:
plt.figure(figsize=(10,5))

freq_x, bin_edges_x = np.histogram(ibtracs['lat'].to_numpy(), range=lat_range, bins=lat_range[1])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=2.0, label=INFO['ibtracs']['label'], color=INFO['ibtracs']['color'])

freq_x, bin_edges_x = np.histogram(owz['lat'].to_numpy(), range=lat_range, bins=lat_range[1])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['owz']['label'], color=INFO['owz']['color'])

freq_x, bin_edges_x = np.histogram(track['lat'].to_numpy(), range=lat_range, bins=lat_range[1])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['track']['label'], color=INFO['track']['color'])

freq_x, bin_edges_x = np.histogram(uz['lat'].to_numpy(), range=lat_range, bins=lat_range[1])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['uz']['label'], color=INFO['uz']['color'])

freq_x, bin_edges_x = np.histogram(cnrm['lat'].to_numpy(), range=lat_range, bins=lat_range[1])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['cnrm']['label'], color=INFO['cnrm']['color'])

freq_x, bin_edges_x = np.histogram(bytestorm['lat'].to_numpy(), range=lat_range, bins=lat_range[1])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=2.0, label=INFO['bytestorm']['label'], color=INFO['bytestorm']['color'])

plt.xlabel('Latitudes', fontdict={'weight':'bold'})
plt.ylabel('Frequencies', fontdict={'weight':'bold'})
plt.xlim(*lat_range)
plt.title(f'Latitude distributions of TC detections')
plt.legend()
outfile = os.path.join(out_dir, f'spatial_dist_lat.{file_ext}') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
plt.figure(figsize=(15,5))

freq_x, bin_edges_x = np.histogram(((ibtracs['lon'] + 360) % 360).to_numpy(), range=lon_range, bins=lon_range[1]-lon_range[0])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=2.0, label=INFO['ibtracs']['label'], color=INFO['ibtracs']['color'])

freq_x, bin_edges_x = np.histogram(((owz['lon'] + 360) % 360).to_numpy(), range=lon_range, bins=lon_range[1]-lon_range[0])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['owz']['label'], color=INFO['owz']['color'])

freq_x, bin_edges_x = np.histogram(((track['lon'] + 360) % 360).to_numpy(), range=lon_range, bins=lon_range[1]-lon_range[0])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['track']['label'], color=INFO['track']['color'])

freq_x, bin_edges_x = np.histogram(((uz['lon'] + 360) % 360).to_numpy(), range=lon_range, bins=lon_range[1]-lon_range[0])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['uz']['label'], color=INFO['uz']['color'])

freq_x, bin_edges_x = np.histogram(((cnrm['lon'] + 360) % 360).to_numpy(), range=lon_range, bins=lon_range[1]-lon_range[0])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['cnrm']['label'], color=INFO['cnrm']['color'])

freq_x, bin_edges_x = np.histogram(((bytestorm['lon'] + 360) % 360).to_numpy(), range=lon_range, bins=lon_range[1]-lon_range[0])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=2.0, label=INFO['bytestorm']['label'], color=INFO['bytestorm']['color'])

plt.xlabel('Longitudes', fontdict={'weight':'bold'})
plt.ylabel('Frequencies', fontdict={'weight':'bold'})
plt.xlim(*lon_range)
plt.title(f'Longitude distributions of TC detections')
plt.legend()
outfile = os.path.join(out_dir, f'spatial_dist_lon.{file_ext}') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

# Track Durations

In [ ]:
plt.figure(figsize=(8,4))

ibtracs_track_durations = ibtracs.track_id.value_counts().to_numpy()//4
bins = (ibtracs_track_durations).max()
freq_x, bin_edges_x = np.histogram(ibtracs_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], drawstyle='steps', linewidth=2.0, label=INFO['ibtracs']['label'], color=INFO['ibtracs']['color'])

owz_track_durations = owz.track_id.value_counts().to_numpy()//4
bins = (owz_track_durations).max()
freq_x, bin_edges_x = np.histogram(owz_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], drawstyle='steps', linewidth=1.0, label=INFO['owz']['label'], color=INFO['owz']['color'], alpha=0.7)

track_track_durations = track.track_id.value_counts().to_numpy()//4
bins = (track_track_durations).max()
freq_x, bin_edges_x = np.histogram(track_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], drawstyle='steps', linewidth=1.0, label=INFO['track']['label'], color=INFO['track']['color'], alpha=0.7)

uz_track_durations = uz.track_id.value_counts().to_numpy()//4
bins = (uz_track_durations).max()
freq_x, bin_edges_x = np.histogram(uz_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], drawstyle='steps', linewidth=1.0, label=INFO['uz']['label'], color=INFO['uz']['color'], alpha=0.7)

cnrm_track_durations = cnrm.track_id.value_counts().to_numpy()//4
bins = (cnrm_track_durations).max()
freq_x, bin_edges_x = np.histogram(cnrm_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], drawstyle='steps', linewidth=1.0, label=INFO['cnrm']['label'], color=INFO['cnrm']['color'], alpha=0.7)

bytestorm_track_durations = bytestorm.track_id.value_counts().to_numpy()//4
bins = (bytestorm_track_durations).max()
freq_x, bin_edges_x = np.histogram(bytestorm_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], drawstyle='steps', linewidth=2.0, label=INFO['bytestorm']['label'], color=INFO['bytestorm']['color'])

plt.xlabel('Track duration (days)', fontdict={'weight':'bold'})
plt.ylabel('Number of TC Tracks', fontdict={'weight':'bold'})
plt.title(f'Track Duration')
plt.legend()

outfile = os.path.join(out_dir, f'track_duration_days.{file_ext}') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
# NOTE: LA LUNGHEZZA DELLE TRACCE E' RIMASTA UGUALE, NON E' CAMBIATO NULLA
# E' IL TIPO DI PLOT CHE LE FA SEMBRARE UN NUMERO INFERIORE DI GIORNI
bytestorm[['track_id']].groupby('track_id').value_counts().values.min()

In [ ]:
plt.figure(figsize=(8,4))

ibtracs_track_durations = ibtracs.track_id.value_counts().to_numpy()//4
bins = (ibtracs_track_durations).max()
freq_x, bin_edges_x = np.histogram(ibtracs_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], drawstyle='steps', linewidth=2.0, label=INFO['ibtracs']['label'], color=INFO['ibtracs']['color'])

owz_track_durations = owz[owz['track_id'].isin(owz_matches['id_owz'].unique())].track_id.value_counts().to_numpy()//4
bins = (owz_track_durations).max()
freq_x, bin_edges_x = np.histogram(owz_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], drawstyle='steps', linewidth=1.0, alpha=0.7, label=INFO['owz']['label'], color=INFO['owz']['color'])

track_track_durations = track[track['track_id'].isin(track_matches['id_track'].unique())].track_id.value_counts().to_numpy()//4
bins = (track_track_durations).max()
freq_x, bin_edges_x = np.histogram(track_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], drawstyle='steps', linewidth=1.0, alpha=0.7, label=INFO['track']['label'], color=INFO['track']['color'])

uz_track_durations = uz[uz['track_id'].isin(uz_matches['id_uz'].unique())].track_id.value_counts().to_numpy()//4
bins = (uz_track_durations).max()
freq_x, bin_edges_x = np.histogram(uz_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], drawstyle='steps', linewidth=1.0, alpha=0.7, label=INFO['uz']['label'], color=INFO['uz']['color'])

cnrm_track_durations = cnrm[cnrm['track_id'].isin(cnrm_matches['id_cnrm'].unique())].track_id.value_counts().to_numpy()//4
bins = (cnrm_track_durations).max()
freq_x, bin_edges_x = np.histogram(cnrm_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], drawstyle='steps', linewidth=1.0, alpha=0.7, label=INFO['cnrm']['label'], color=INFO['cnrm']['color'])

bytestorm_track_durations = bytestorm[bytestorm['track_id'].isin(bytestorm_matches['id_'+our_work_name].unique())].track_id.value_counts().to_numpy()//4
bins = (bytestorm_track_durations).max()
freq_x, bin_edges_x = np.histogram(bytestorm_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], drawstyle='steps', linewidth=2.0, label=INFO['bytestorm']['label'], color=INFO['bytestorm']['color'])

plt.xlabel('Track duration (days)', fontdict={'weight':'bold'})
# plt.yticks([])
# plt.ylabel("")
# plt.xlim(-1, 40)
# plt.ylim(0, 600)
plt.title(f'Track Duration')
plt.legend()

outfile = os.path.join(out_dir, f'track_duration_days_on_matched_tcs.{file_ext}') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
owz_false_alarms = owz[~owz['track_id'].isin(owz_matches['id_owz'].unique())]
owz_true_positives = owz[owz['track_id'].isin(owz_matches['id_owz'].unique())]

track_false_alarms = track[~track['track_id'].isin(track_matches['id_track'].unique())]
track_true_positives = track[track['track_id'].isin(track_matches['id_track'].unique())]

uz_false_alarms = uz[~uz['track_id'].isin(uz_matches['id_uz'].unique())]
uz_true_positives = uz[uz['track_id'].isin(uz_matches['id_uz'].unique())]

cnrm_false_alarms = cnrm[~cnrm['track_id'].isin(cnrm_matches['id_cnrm'].unique())]
cnrm_true_positives = cnrm[cnrm['track_id'].isin(cnrm_matches['id_cnrm'].unique())]

bytestorm_false_alarms = bytestorm[~bytestorm['track_id'].isin(bytestorm_matches['id_'+our_work_name].unique())]
bytestorm_true_positives = bytestorm[bytestorm['track_id'].isin(bytestorm_matches['id_'+our_work_name].unique())]

In [ ]:
plt.figure(figsize=(8,4))

# ibtracs_track_durations = ibtracs.track_id.value_counts().to_numpy()//4
# bins = (ibtracs_track_durations).max()
# freq_x, bin_edges_x = np.histogram(ibtracs_track_durations, bins=bins, range=(3,bins+3))
# plt.plot(bin_edges_x[:-3], freq_x[:-2], label='IBTrACS', linewidth=2.0, color='tab:blue')

owz_fp_track_durations = owz_false_alarms.track_id.value_counts().to_numpy()//4
bins = (owz_fp_track_durations).max()
freq_x, bin_edges_x = np.histogram(owz_fp_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], label=INFO['owz']['label'], linewidth=1.0, color=INFO['owz']['color'], drawstyle='steps-post')

track_fp_track_durations = track_false_alarms.track_id.value_counts().to_numpy()//4
bins = (track_fp_track_durations).max()
freq_x, bin_edges_x = np.histogram(track_fp_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], label=INFO['track']['label'], linewidth=1.0, color=INFO['track']['color'], drawstyle='steps-post')

uz_fp_track_durations = uz_false_alarms.track_id.value_counts().to_numpy()//4
bins = (uz_fp_track_durations).max()
freq_x, bin_edges_x = np.histogram(uz_fp_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], label=INFO['uz']['label'], linewidth=1.0, color=INFO['uz']['color'], drawstyle='steps-post')

cnrm_fp_track_durations = cnrm_false_alarms.track_id.value_counts().to_numpy()//4
bins = (cnrm_fp_track_durations).max()
freq_x, bin_edges_x = np.histogram(cnrm_fp_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], label=INFO['cnrm']['label'], linewidth=1.0, color=INFO['cnrm']['color'], drawstyle='steps-post')

bytestorm_fp_track_durations = bytestorm_false_alarms.track_id.value_counts().to_numpy()//4
bins = (bytestorm_fp_track_durations).max()
freq_x, bin_edges_x = np.histogram(bytestorm_fp_track_durations, bins=bins, range=(3,bins+3))
plt.plot(bin_edges_x[:-3], freq_x[:-2], label=INFO['bytestorm']['label'], linewidth=2.0, color=INFO['bytestorm']['color'], drawstyle='steps-post')

plt.xlabel('Duration [days]', fontdict={'weight':'bold'})
plt.title(f'Track Duration of False Positives')
plt.legend()


outfile = os.path.join(out_dir, f'track_duration_days_on_false_alarms.{file_ext}') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

# Tropical Storm Track Density

It is computed as the storm transit per month on each point in the domain

In [ ]:
lats, lons = np.arange(-90, 90.25, 0.25)[::-1], np.arange(0, 360.25, 0.25)
degs = 2
grid_res = 0.25
bins = [int(len(lats) // (1 / grid_res) // degs), int(len(lons) // (1 / grid_res) // degs)]
map_range = [[0,len(lats)],[0,len(lons)]] # extent of the original map
norm_month = False

ibtracs_storm_transits = compute_storm_transits(ibtracs, bins, map_range, lats, lons, norm_month)
owz_storm_transits = compute_storm_transits(owz, bins, map_range, lats, lons, norm_month)
track_storm_transits = compute_storm_transits(track, bins, map_range, lats, lons, norm_month)
uz_storm_transits = compute_storm_transits(uz, bins, map_range, lats, lons, norm_month)
cnrm_storm_transits = compute_storm_transits(cnrm, bins, map_range, lats, lons, norm_month)
bytestorm_storm_transits = compute_storm_transits(bytestorm, bins, map_range, lats, lons, norm_month)

vmin = np.nanmin((
    ibtracs_storm_transits,
    # owz_storm_transits,
    # track_storm_transits,
    # uz_storm_transits,
    # cnrm_storm_transits,
    # bytestorm_storm_transits,
))
vmax = np.nanmax((
    ibtracs_storm_transits,
    # owz_storm_transits,
    # track_storm_transits,
    # uz_storm_transits,
    # cnrm_storm_transits,
    # bytestorm_storm_transits,
))

plot_storm_track_density(ibtracs_storm_transits, 'ibtracs', vmin, vmax, lat_range, lon_range, cmap='jet', grid_spacing=20)
outfile = os.path.join(out_dir, f'storm_transits_ibtracs.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

plot_storm_track_density(owz_storm_transits, 'owz', vmin, vmax, lat_range, lon_range, cmap='jet', grid_spacing=20)
outfile = os.path.join(out_dir, f'storm_transits_owz.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

plot_storm_track_density(track_storm_transits, 'track', vmin, vmax, lat_range, lon_range, cmap='jet', grid_spacing=20)
outfile = os.path.join(out_dir, f'storm_transits_track.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

plot_storm_track_density(uz_storm_transits, 'uz', vmin, vmax, lat_range, lon_range, cmap='jet', grid_spacing=20)
outfile = os.path.join(out_dir, f'storm_transits_uz.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

plot_storm_track_density(cnrm_storm_transits, 'cnrm', vmin, vmax, lat_range, lon_range, cmap='jet', grid_spacing=20)
outfile = os.path.join(out_dir, f'storm_transits_cnrm.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

plot_storm_track_density(bytestorm_storm_transits, our_work_name, vmin, vmax, lat_range, lon_range, cmap='jet', grid_spacing=20)
outfile = os.path.join(out_dir, f'storm_transits_bytestorm.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
lats, lons = np.arange(-90, 90.25, 0.25)[::-1], np.arange(0, 360.25, 0.25)
degs = 2
grid_res = 0.25
bins = [int(len(lats) // (1 / grid_res) // degs), int(len(lons) // (1 / grid_res) // degs)]
map_range = [[0,len(lats)],[0,len(lons)]] # extent of the original map
norm_month = False

owz_true_positive_storm_transits = compute_storm_transits(owz_true_positives, bins, map_range, lats, lons, norm_month)
track_true_positive_storm_transits = compute_storm_transits(track_true_positives, bins, map_range, lats, lons, norm_month)
uz_true_positive_storm_transits = compute_storm_transits(uz_true_positives, bins, map_range, lats, lons, norm_month)
cnrm_true_positive_storm_transits = compute_storm_transits(cnrm_true_positives, bins, map_range, lats, lons, norm_month)
bytestorm_true_positive_storm_transits = compute_storm_transits(bytestorm_true_positives, bins, map_range, lats, lons, norm_month)

vmin, vmax = np.nanmin((owz_true_positive_storm_transits)), np.nanmax((owz_true_positive_storm_transits))
plot_storm_track_density(owz_true_positive_storm_transits, INFO['owz']['label'], vmin, vmax, lat_range, lon_range, cmap='jet')
outfile = os.path.join(out_dir, f'storm_transits_owz_TP.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

vmin, vmax = np.nanmin((track_true_positive_storm_transits)), np.nanmax((track_true_positive_storm_transits))
plot_storm_track_density(track_true_positive_storm_transits, INFO['track']['label'], vmin, vmax, lat_range, lon_range, cmap='jet')
outfile = os.path.join(out_dir, f'storm_transits_track_TP.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

vmin, vmax = np.nanmin((uz_true_positive_storm_transits)), np.nanmax((uz_true_positive_storm_transits))
plot_storm_track_density(uz_true_positive_storm_transits, INFO['uz']['label'], vmin, vmax, lat_range, lon_range, cmap='jet')
outfile = os.path.join(out_dir, f'storm_transits_uz_TP.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

vmin, vmax = np.nanmin((cnrm_true_positive_storm_transits)), np.nanmax((cnrm_true_positive_storm_transits))
plot_storm_track_density(cnrm_true_positive_storm_transits, INFO['cnrm']['label'], vmin, vmax, lat_range, lon_range, cmap='jet')
outfile = os.path.join(out_dir, f'storm_transits_cnrm_TP.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

vmin, vmax = np.nanmin((bytestorm_true_positive_storm_transits)), np.nanmax((bytestorm_true_positive_storm_transits))
plot_storm_track_density(bytestorm_true_positive_storm_transits, INFO['bytestorm']['label'], vmin, vmax, lat_range, lon_range, cmap='jet')
outfile = os.path.join(out_dir, f'storm_transits_bytestorm_TP.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
lats, lons = np.arange(-90, 90.25, 0.25)[::-1], np.arange(0, 360.25, 0.25)
degs = 2
grid_res = 0.25
bins = [int(len(lats) // (1 / grid_res) // degs), int(len(lons) // (1 / grid_res) // degs)]
map_range = [[0,len(lats)],[0,len(lons)]] # extent of the original map
norm_month = False

owz_false_alarms_storm_transits = compute_storm_transits(owz_false_alarms, bins, map_range, lats, lons, norm_month)
track_false_alarms_storm_transits = compute_storm_transits(track_false_alarms, bins, map_range, lats, lons, norm_month)
uz_false_alarms_storm_transits = compute_storm_transits(uz_false_alarms, bins, map_range, lats, lons, norm_month)
cnrm_false_alarms_storm_transits = compute_storm_transits(cnrm_false_alarms, bins, map_range, lats, lons, norm_month)
bytestorm_false_alarms_storm_transits = compute_storm_transits(bytestorm_false_alarms, bins, map_range, lats, lons, norm_month)

vmin, vmax = np.nanmin((owz_false_alarms_storm_transits)), np.nanmax((owz_false_alarms_storm_transits))
plot_storm_track_density(owz_false_alarms_storm_transits, INFO['owz']['label'], vmin, vmax, lat_range, lon_range, cmap='jet')
outfile = os.path.join(out_dir, f'storm_transits_owz_FA.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

vmin, vmax = np.nanmin((track_false_alarms_storm_transits)), np.nanmax((track_false_alarms_storm_transits))
plot_storm_track_density(track_false_alarms_storm_transits, INFO['track']['label'], vmin, vmax, lat_range, lon_range, cmap='jet')
outfile = os.path.join(out_dir, f'storm_transits_track_FA.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

vmin, vmax = np.nanmin((uz_false_alarms_storm_transits)), np.nanmax((uz_false_alarms_storm_transits))
plot_storm_track_density(uz_false_alarms_storm_transits, INFO['uz']['label'], vmin, vmax, lat_range, lon_range, cmap='jet')
outfile = os.path.join(out_dir, f'storm_transits_uz_FA.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

vmin, vmax = np.nanmin((cnrm_false_alarms_storm_transits)), np.nanmax((cnrm_false_alarms_storm_transits))
plot_storm_track_density(cnrm_false_alarms_storm_transits, INFO['cnrm']['label'], vmin, vmax, lat_range, lon_range, cmap='jet')
outfile = os.path.join(out_dir, f'storm_transits_cnrm_FA.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

vmin, vmax = np.nanmin((bytestorm_false_alarms_storm_transits)), np.nanmax((bytestorm_false_alarms_storm_transits))
plot_storm_track_density(bytestorm_false_alarms_storm_transits, INFO['bytestorm']['label'], vmin, vmax, lat_range, lon_range, cmap='jet')
outfile = os.path.join(out_dir, f'storm_transits_bytestorm_FA.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
transits = [ibtracs_storm_transits, bytestorm_true_positive_storm_transits, bytestorm_false_alarms_storm_transits]
shape = bytestorm_false_alarms_storm_transits.shape
grid_spacing = 20
cmap = 'jet'
lon_grid, lat_grid = np.meshgrid(np.linspace(*lon_range, shape[1]), np.linspace(*lat_range, shape[0])[::-1])
central_longitude = (lon_range[1] - lon_range[0]) // 2
image_extent = [lon_range[0], lon_range[1], lat_range[0], lat_range[1]]

proj = ccrs.PlateCarree(central_longitude=central_longitude)
fig, axes = plt.subplots(1, 1, figsize=(9,9))
axes.axis('off')
gs = plt.GridSpec(3, 1, figure=fig)

fontdict = {'size':12}
title_fontdict = {'size':16}
ticksize = 10

vmax = np.nanmax(transits[0:2])
vmin = np.nanmin(transits[0:2])
ax = fig.add_subplot(gs[0], projection=proj)
ax.set_extent(image_extent, crs=ccrs.PlateCarree())
ax.coastlines(resolution='50m', lw=0.2)
ax.add_feature(cf.LAND, facecolor='lightgrey', alpha=0.3)
im = ax.pcolormesh(lon_grid, lat_grid, transits[0], cmap=cmap, transform=ccrs.PlateCarree(), vmin=vmin, vmax=vmax)
plt.colorbar(im, ax=ax)
# x-axis
longitudes = np.arange(lon_range[0], lon_range[1]+1, grid_spacing)
lon_formatter = LongitudeFormatter(zero_direction_label=False)
ax.set_xticklabels([])
ax.set_xticks(longitudes-central_longitude)
# y-axis
latitudes = np.arange(lat_range[0], lat_range[1]+1, grid_spacing)
lat_formatter = LatitudeFormatter()
ax.yaxis.set_major_formatter(lat_formatter)
ax.yaxis.set_major_locator(mticker.FixedLocator(latitudes))
ax.set_yticklabels(latitudes, size=ticksize)
ax.set_yticks(latitudes)
ax.set_ylabel('Latitude [deg]', fontdict=fontdict)
# gridlines
gl = ax.gridlines(crs=proj, draw_labels=False, dms=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.xlocator = mticker.FixedLocator(longitudes-central_longitude)
gl.ylocator = mticker.FixedLocator(latitudes)
gl.xlines = True
gl.ylines = True

ax = fig.add_subplot(gs[1], projection=proj)
ax.set_extent(image_extent, crs=ccrs.PlateCarree())
ax.coastlines(resolution='50m', lw=0.2)
ax.add_feature(cf.LAND, facecolor='lightgrey', alpha=0.3)
im = ax.pcolormesh(lon_grid, lat_grid, transits[1], cmap=cmap, transform=ccrs.PlateCarree(), vmin=vmin, vmax=vmax)
plt.colorbar(im, ax=ax)
# x-axis
longitudes = np.arange(lon_range[0], lon_range[1]+1, grid_spacing)
lon_formatter = LongitudeFormatter(zero_direction_label=False)
ax.set_xticklabels([])
ax.set_xticks(longitudes-central_longitude)
# y-axis
latitudes = np.arange(lat_range[0], lat_range[1]+1, grid_spacing)
lat_formatter = LatitudeFormatter()
ax.yaxis.set_major_formatter(lat_formatter)
ax.yaxis.set_major_locator(mticker.FixedLocator(latitudes))
ax.set_yticklabels(latitudes, size=ticksize)
ax.set_yticks(latitudes)
ax.set_ylabel('Latitude [deg]', fontdict=fontdict)
# gridlines
gl = ax.gridlines(crs=proj, draw_labels=False, dms=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.xlocator = mticker.FixedLocator(longitudes-central_longitude)
gl.ylocator = mticker.FixedLocator(latitudes)
gl.xlines = True
gl.ylines = True

vmax = np.nanmax(transits[2])
vmin = np.nanmin(transits[2])
ax = fig.add_subplot(gs[2], projection=proj)
ax.set_extent(image_extent, crs=ccrs.PlateCarree())
ax.coastlines(resolution='50m', lw=0.2)
ax.add_feature(cf.LAND, facecolor='lightgrey', alpha=0.3)
im = ax.pcolormesh(lon_grid, lat_grid, transits[2], cmap=cmap, transform=ccrs.PlateCarree(), vmin=vmin, vmax=vmax)
plt.colorbar(im, ax=ax)
# x-axis
longitudes = np.arange(lon_range[0], lon_range[1]+1, grid_spacing)
lon_formatter = LongitudeFormatter(zero_direction_label=False)
ax.xaxis.set_major_formatter(lon_formatter)
ax.xaxis.set_major_locator(mticker.FixedLocator(longitudes-central_longitude))
ax.set_xticklabels(longitudes, size=ticksize)
ax.set_xticks(longitudes-central_longitude)
ax.set_xlabel('Longitude [deg]', fontdict=fontdict)
# y-axis
latitudes = np.arange(lat_range[0], lat_range[1]+1, grid_spacing)
lat_formatter = LatitudeFormatter()
ax.yaxis.set_major_formatter(lat_formatter)
ax.yaxis.set_major_locator(mticker.FixedLocator(latitudes))
ax.set_yticklabels(latitudes, size=ticksize)
ax.set_yticks(latitudes)
ax.set_ylabel('Latitude [deg]', fontdict=fontdict)
# gridlines
gl = ax.gridlines(crs=proj, draw_labels=False, dms=True, linewidth=0.5, color='gray', alpha=0.5, linestyle='--')
gl.xlocator = mticker.FixedLocator(longitudes-central_longitude)
gl.ylocator = mticker.FixedLocator(latitudes)
gl.xlines = True
gl.ylines = True

# plt.tight_layout()
outfile = os.path.join(out_dir, f'storm_transits_bytestorm_TP_FA.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

## Tropical Storm FPs Evolution Gifs

In [ ]:
if compute_gif:
    # Show only these TIDs here
    tids = [
        ('OT_27_(18-39)', 8, 6),
        ('OT_95_(63-80)', 8, 8),
        ('OT_121_(81-114)', 8, 20),
    ]

In [ ]:
if compute_gif:
    src = '/Users/davide/Developer/tropical_cyclone_tracking/data/datasets/global'

    from tqdm import tqdm
    for tid in list(bytestorm_false_alarms.track_id.unique()[:3]):
        for item in tqdm(list(bytestorm_false_alarms[bytestorm_false_alarms['track_id']==tid].time.unique())):
            filename = f'{item.year}_{item.month:02d}_{item.day:02d}_{item.hour:02d}.nc'
            remote_src = f'/work/cmcc/machine_learning/dd26322/tropical_cyclone_tracking/data/datasets/global/{filename}'
            local_src = f'/Users/davide/Developer/tropical_cyclone_tracking/data/datasets/global'
            local_dst = os.path.join(local_src, filename)
            # print(filename)
            if os.path.exists(local_dst): continue
            os.system(f'scp -r dd26322@login1.juno.cmcc.scc:{remote_src} {local_src}')

In [ ]:
def get_filenames(src, tid):
    dates = bytestorm[bytestorm['track_id']==tid].time.unique()
    fnames = []
    for date in dates:
        fnames.append(os.path.join(src, f'{date.year}_{date.month:02d}_{date.day:02d}_{date.hour:02d}.nc'))
    return fnames

In [ ]:
if compute_gif:
    tid, row, col = tids[0]

    ds = xr.open_mfdataset(get_filenames(src, tid))
    pds = ds.coarsen({'lat':40, 'lon':40}, boundary="trim").construct({'lon':("cols", "lon_range"), 'lat':("rows", "lat_range")})
    fig, axes = plt.subplots(1,2,figsize=(12,4))

    def animate(i):
        pds.sel(rows=row, cols=col).isel(time=i)['vo_850'].plot(ax=axes[0], add_colorbar=False, yincrease=False)
        pds.sel(rows=row, cols=col).isel(time=i)['msl'].plot(ax=axes[1], add_colorbar=False, yincrease=False, cmap='RdBu_r')

    ani = animation.FuncAnimation(fig, animate, len(pds.time.values), interval=400, blit=False)

    im = pds.sel(rows=row, cols=col).isel(time=0)['vo_850'].plot(ax=axes[0], add_colorbar=False, yincrease=False)
    cbar = fig.colorbar(im)
    cbar.set_label('Vorticity at 850mb')
    im = pds.sel(rows=row, cols=col).isel(time=0)['msl'].plot(ax=axes[1], add_colorbar=False, yincrease=False, cmap='RdBu_r')
    cbar = fig.colorbar(im)
    cbar.set_label('MSLP')
    fig.suptitle(y=1.0, t=f"{tid} Track", fontsize=15)

    outfile = os.path.join(out_dir, f'anim_{tid}.gif') if save_figures else None
    ani.save(outfile, writer='pillow', fps=1) if outfile else None
    HTML(ani.to_jshtml()) # Show the animation in the kernel

In [ ]:
if compute_gif:
    tid, row, col = tids[1]

    ds = xr.open_mfdataset(get_filenames(src, tid))
    pds = ds.coarsen({'lat':40, 'lon':40}, boundary="trim").construct({'lon':("cols", "lon_range"), 'lat':("rows", "lat_range")})
    fig, axes = plt.subplots(1,2,figsize=(12,4))

    def animate(i):
        pds.sel(rows=row, cols=col).isel(time=i)['vo_850'].plot(ax=axes[0], add_colorbar=False, yincrease=False)
        pds.sel(rows=row, cols=col).isel(time=i)['msl'].plot(ax=axes[1], add_colorbar=False, yincrease=False, cmap='RdBu_r')

    ani = animation.FuncAnimation(fig, animate, len(pds.time.values), interval=400, blit=False)

    im = pds.sel(rows=row, cols=col).isel(time=0)['vo_850'].plot(ax=axes[0], add_colorbar=False, yincrease=False)
    cbar = fig.colorbar(im)
    cbar.set_label('Vorticity at 850mb')
    im = pds.sel(rows=row, cols=col).isel(time=0)['msl'].plot(ax=axes[1], add_colorbar=False, yincrease=False, cmap='RdBu_r')
    cbar = fig.colorbar(im)
    cbar.set_label('MSLP')
    fig.suptitle(y=1.0, t=f"{tid} Track", fontsize=15)

    outfile = os.path.join(out_dir, f'anim_{tid}.gif') if save_figures else None
    ani.save(outfile, writer='pillow', fps = 1) if outfile else None
    HTML(ani.to_jshtml()) # Show the animation in the kernel

In [ ]:
if compute_gif:
    tid, row, col = tids[2]

    ds = xr.open_mfdataset(get_filenames(src, tid))
    pds = ds.coarsen({'lat':40, 'lon':40}, boundary="trim").construct({'lon':("cols", "lon_range"), 'lat':("rows", "lat_range")})
    fig, axes = plt.subplots(1,2,figsize=(12,4))

    def animate(i):
        pds.sel(rows=row, cols=col).isel(time=i)['vo_850'].plot(ax=axes[0], add_colorbar=False, yincrease=False)
        pds.sel(rows=row, cols=col).isel(time=i)['msl'].plot(ax=axes[1], add_colorbar=False, yincrease=False, cmap='RdBu_r')

    ani = animation.FuncAnimation(fig, animate, len(pds.time.values), interval=400, blit=False)

    im = pds.sel(rows=row, cols=col).isel(time=0)['vo_850'].plot(ax=axes[0], add_colorbar=False, yincrease=False)
    cbar = fig.colorbar(im)
    cbar.set_label('Vorticity at 850mb')
    im = pds.sel(rows=row, cols=col).isel(time=0)['msl'].plot(ax=axes[1], add_colorbar=False, yincrease=False, cmap='RdBu_r')
    cbar = fig.colorbar(im)
    cbar.set_label('MSLP')
    fig.suptitle(y=1.0, t=f"{tid} Track", fontsize=15)

    outfile = os.path.join(out_dir, f'anim_{tid}.gif') if save_figures else None
    ani.save(outfile, writer='pillow', fps = 1) if outfile else None
    HTML(ani.to_jshtml()) # Show the animation in the kernel

In [ ]:
if compute_gif:
    i = 0
    for tid in tids:
        # print(tid)
        plot_tracks(bytestorm_false_alarms[bytestorm_false_alarms.track_id==tid], None, lat_range, lon_range, f'{tid}', None)
        i+=1
        if i == 10:
            break

# Seasonality

Plot number of TC tracks for every month of year

- Maybe create a plot for each Tracker
- Maybe add only False Alarms and Hits (as Bourdin)

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

freq_x, bin_edges_x = np.histogram(ibtracs.time.dt.month, bins=12, range=[1,13])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=2.0, label=INFO['ibtracs']['label'], color=INFO['ibtracs']['color'])

freq_x, bin_edges_x = np.histogram(owz.time.dt.month, bins=12, range=[1,13])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['owz']['label'], color=INFO['owz']['color'])

freq_x, bin_edges_x = np.histogram(track.time.dt.month, bins=12, range=[1,13])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['track']['label'], color=INFO['track']['color'])

freq_x, bin_edges_x = np.histogram(uz.time.dt.month, bins=12, range=[1,13])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['uz']['label'], color=INFO['uz']['color'])

freq_x, bin_edges_x = np.histogram(cnrm.time.dt.month, bins=12, range=[1,13])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['cnrm']['label'], color=INFO['cnrm']['color'])

freq_x, bin_edges_x = np.histogram(bytestorm.time.dt.month, bins=12, range=[1,13])
plt.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=2.0, label=INFO['bytestorm']['label'], color=INFO['bytestorm']['color'])

plt.ylim([0,5000])
plt.xlabel('Month')
plt.ylabel('#TC detections')
ax.set_xticks(np.arange(1, 13, 1))
ax.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
ax.spines[['right', 'top']].set_visible(False)
plt.title(f'Seasonal Tropical Cyclone distribution')
leg = fig.legend(loc='upper center', markerscale=12, fontsize=10, edgecolor='gray', framealpha=1, ncol=6, bbox_to_anchor=(0.5, 0.02))
for legobj in leg.legend_handles:
    legobj.set_linewidth(2.0)

outfile = os.path.join(out_dir, f'seasonal_tc_dist.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
def clean_tracks_for_gen_points(tracks: pd.DataFrame):
    # Geographical details
    tracks.loc[tracks.lon < 0, "lon"] += 360
    tracks["hemisphere"] = np.where(tracks.lat > 0, "N", "S")
    tracks["basin"] = dynamicopy.get_basin(tracks.lon, tracks.lat)

    # All about time
    tracks["hour"] = tracks.time.dt.hour
    tracks["day"] = tracks.time.dt.day
    tracks["month"] = tracks.time.dt.month
    tracks["year"] = tracks.time.dt.year
    tracks = dynamicopy.add_season(tracks)

    # Reset the index
    tracks = tracks.reset_index(drop=True)
    return tracks

In [ ]:
fig = plt.figure(layout='constrained', figsize=(10,4))
gs = fig.add_gridspec(gs_rows, gs_cols)

for i,basin in enumerate([b for b in basins if b not in excluded_basins]):
    gsi = gsix(gs,i)
    axi = fig.add_subplot(gsi)
    max_x = 0.0

    freq_x, bin_edges_x = np.histogram(dynamicopy.genesis_points(clean_tracks_for_gen_points(owz[owz['basin']==basin])).time.dt.month, bins=12, range=[1,13])
    max_x = np.max((np.max(freq_x), max_x))
    axi.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['owz']['label'], color=INFO['owz']['color'])
    freq_x, bin_edges_x = np.histogram(dynamicopy.genesis_points(clean_tracks_for_gen_points(track[track['basin']==basin])).time.dt.month, bins=12, range=[1,13])
    max_x = np.max((np.max(freq_x), max_x))
    axi.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['track']['label'], color=INFO['track']['color'])
    freq_x, bin_edges_x = np.histogram(dynamicopy.genesis_points(clean_tracks_for_gen_points(uz[uz['basin']==basin])).time.dt.month, bins=12, range=[1,13])
    max_x = np.max((np.max(freq_x), max_x))
    axi.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['uz']['label'], color=INFO['uz']['color'])
    freq_x, bin_edges_x = np.histogram(dynamicopy.genesis_points(clean_tracks_for_gen_points(cnrm[cnrm['basin']==basin])).time.dt.month, bins=12, range=[1,13])
    max_x = np.max((np.max(freq_x), max_x))
    axi.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=1.0, label=INFO['cnrm']['label'], color=INFO['cnrm']['color'])
    freq_x, bin_edges_x = np.histogram(dynamicopy.genesis_points(clean_tracks_for_gen_points(bytestorm[bytestorm['basin']==basin])).time.dt.month, bins=12, range=[1,13])
    max_x = np.max((np.max(freq_x), max_x))
    axi.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linewidth=2.0, label=INFO['bytestorm']['label'], color=INFO['bytestorm']['color'])
    freq_x, bin_edges_x = np.histogram(dynamicopy.genesis_points(clean_tracks_for_gen_points(ibtracs[ibtracs['basin']==basin])).time.dt.month, bins=12, range=[1,13])
    max_x = np.max((np.max(freq_x), max_x))
    axi.plot(bin_edges_x[:-1], freq_x, drawstyle='steps', linestyle='dashed', linewidth=2.0, label=INFO['ibtracs']['label'], color=INFO['ibtracs']['color'])

    axi.text(0.6, max_x, basin, fontsize=14, weight='bold', color='black', ha='left', va='center')
    axi.spines[['right', 'top']].set_visible(False)
    if not i: handles, labels = axi.get_legend_handles_labels()
    if i == 0 or i == 3:
        axi.set_ylabel('#TC genesis')
    if i >= 3:
        axi.set_xticks(np.arange(1, 13, 1))
        axi.set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
        axi.set_xlabel('Month')
    else:
        axi.set_xticks(np.arange(1, 13, 1))
        axi.set_xticklabels([])

plt.suptitle(f'Seasonal Tropical Cyclone distribution')
leg = fig.legend(handles, labels, loc='upper center', markerscale=12, fontsize=10, edgecolor='gray', framealpha=1, ncol=6, bbox_to_anchor=(0.5, 0.0))
for legobj in leg.legend_handles:
    legobj.set_linewidth(2.0)

outfile = os.path.join(out_dir, f'seasonal_tc_dist_by_basin.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
# fig = plt.figure(layout='constrained', figsize=gs_figsize)
# gs = fig.add_gridspec(gs_rows, gs_cols)
# for i,basin in enumerate([b for b in basins if b not in excluded_basins]):
#     gsi = gsix(gs,i)
#     axi = fig.add_subplot(gsi)
#     algo_results_bo = pd.concat([owz_res_dict_bo[basin], track_res_dict_bo[basin], uz_res_dict_bo[basin], cnrm_res_dict_bo[basin], bytestorm_res_dict_bo[basin]]).reset_index(drop=True)
#     plot_pod_and_far_on_axis(
#         algo_results_bo,
#         label=f' ({years[0]}-{years[-1]})',
#         label_size=16,
#         ticks_size=14,
#         ax=axi,
#         set_xlabel=True if i > 3 else False,
#         outfile=None,
#     )
#     axi.text(-0.5, 100, basin, fontsize=20, weight='bold', color='black', ha='left', va='center')
#     axi.spines[['right', 'top']].set_visible(False)
#     if not i: handles, labels = axi.get_legend_handles_labels()
# fig.legend(handles, labels, loc='upper center', markerscale=12, edgecolor='gray', fontsize=16, framealpha=1, ncol=4, bbox_to_anchor=(0.5, 0))
# plt.tight_layout()
# outfile = os.path.join(out_dir, f'pod_far_by_basin_bourdin.{file_ext}')
# plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
drawstyle = 'steps'

fig, ax = plt.subplots(1,2,figsize=(12, 5), sharey=True)

freq_x, bin_edges_x = np.histogram(ibtracs.time.dt.month, bins=12, range=[1,13])
ax[0].plot(bin_edges_x[:-1], freq_x, drawstyle=drawstyle, linewidth=2.0, label=INFO['ibtracs']['label'], color=INFO['ibtracs']['color'])

freq_x, bin_edges_x = np.histogram(owz_true_positives.time.dt.month, bins=12, range=[1,13])
ax[0].plot(bin_edges_x[:-1], freq_x, drawstyle=drawstyle, linewidth=1.0, label=INFO['owz']['label'], color=INFO['owz']['color'])

freq_x, bin_edges_x = np.histogram(track_true_positives.time.dt.month, bins=12, range=[1,13])
ax[0].plot(bin_edges_x[:-1], freq_x, drawstyle=drawstyle, linewidth=1.0, label=INFO['track']['label'], color=INFO['track']['color'])

freq_x, bin_edges_x = np.histogram(uz_true_positives.time.dt.month, bins=12, range=[1,13])
ax[0].plot(bin_edges_x[:-1], freq_x, drawstyle=drawstyle, linewidth=1.0, label=INFO['uz']['label'], color=INFO['uz']['color'])

freq_x, bin_edges_x = np.histogram(cnrm_true_positives.time.dt.month, bins=12, range=[1,13])
ax[0].plot(bin_edges_x[:-1], freq_x, drawstyle=drawstyle, linewidth=1.0, label=INFO['cnrm']['label'], color=INFO['cnrm']['color'])

freq_x, bin_edges_x = np.histogram(bytestorm_true_positives.time.dt.month, bins=12, range=[1,13])
ax[0].plot(bin_edges_x[:-1], freq_x, drawstyle=drawstyle, linewidth=2.0, label=INFO['bytestorm']['label'], color=INFO['bytestorm']['color'])


freq_x, bin_edges_x = np.histogram(owz_false_alarms.time.dt.month, bins=12, range=[1,13])
ax[1].plot(bin_edges_x[:-1], freq_x, drawstyle=drawstyle, linewidth=1.0, color=INFO['owz']['color'], linestyle='dashed')

freq_x, bin_edges_x = np.histogram(track_false_alarms.time.dt.month, bins=12, range=[1,13])
ax[1].plot(bin_edges_x[:-1], freq_x, drawstyle=drawstyle, linewidth=1.0, color=INFO['track']['color'], linestyle='dashed')

freq_x, bin_edges_x = np.histogram(uz_false_alarms.time.dt.month, bins=12, range=[1,13])
ax[1].plot(bin_edges_x[:-1], freq_x, drawstyle=drawstyle, linewidth=1.0, color=INFO['uz']['color'], linestyle='dashed')

freq_x, bin_edges_x = np.histogram(cnrm_false_alarms.time.dt.month, bins=12, range=[1,13])
ax[1].plot(bin_edges_x[:-1], freq_x, drawstyle=drawstyle, linewidth=1.0, color=INFO['cnrm']['color'], linestyle='dashed')

freq_x, bin_edges_x = np.histogram(bytestorm_false_alarms.time.dt.month, bins=12, range=[1,13])
ax[1].plot(bin_edges_x[:-1], freq_x, drawstyle=drawstyle, linewidth=2.0, color=INFO['bytestorm']['color'], linestyle='dashed')

ax[0].set_xlabel('Month')
ax[1].set_xlabel('Month')
ax[0].set_ylabel('#TC detections')

ax[0].set_xticks(np.arange(1, 13, 1))
ax[1].set_xticks(np.arange(1, 13, 1))
ax[0].set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])
ax[1].set_xticklabels(['J', 'F', 'M', 'A', 'M', 'J', 'J', 'A', 'S', 'O', 'N', 'D'])

plt.subplots_adjust(wspace=0.05, hspace=0)
ax[0].set_title('Hits')
ax[1].set_title('False Alarms')
plt.suptitle(f'Seasonal Tropical Cyclone distribution')

ax[0].spines[['right', 'top']].set_visible(False)
ax[1].spines[['right', 'top']].set_visible(False)

handles, labels = ax[0].get_legend_handles_labels()
leg = fig.legend(handles, labels, loc='upper center', markerscale=12, edgecolor='gray', framealpha=1, ncol=6, bbox_to_anchor=(0.5, 0.03))
for legobj in leg.legend_handles:
    legobj.set_linewidth(2.0)
plt.tight_layout()
outfile = os.path.join(out_dir, f'seasonal_tc_dist_TP_FA.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

# Interannual Variability

Plot the number of TC tracks for every year

In [ ]:
# compute interannual variabilities
ibtracs['year'] = ibtracs['time'].dt.year
owz['year'] = owz['time'].dt.year
track['year'] = track['time'].dt.year
uz['year'] = uz['time'].dt.year
cnrm['year'] = cnrm['time'].dt.year
bytestorm['year'] = bytestorm['time'].dt.year

years_df = pd.DataFrame(data={'year':years})

ibtracs_iav = dict()
owz_iav = dict()
track_iav = dict()
uz_iav = dict()
cnrm_iav = dict()
bytestorm_iav = dict()
for basin in basins:
    ibtracs_iav[basin] = pd.merge(ibtracs[ibtracs['basin']==basin].groupby(by='year')['track_id'].unique().apply(len).reset_index(), years_df, on='year', how='right').fillna(0)['track_id'].to_numpy()
    owz_iav[basin] = pd.merge(owz[owz['basin']==basin].groupby(by='year')['track_id'].unique().apply(len).reset_index(), years_df, on='year', how='right').fillna(0)['track_id'].to_numpy()
    track_iav[basin] = pd.merge(track[track['basin']==basin].groupby(by='year')['track_id'].unique().apply(len).reset_index(), years_df, on='year', how='right').fillna(0)['track_id'].to_numpy()
    uz_iav[basin] = pd.merge(uz[uz['basin']==basin].groupby(by='year')['track_id'].unique().apply(len).reset_index(), years_df, on='year', how='right').fillna(0)['track_id'].to_numpy()
    cnrm_iav[basin] = pd.merge(cnrm[cnrm['basin']==basin].groupby(by='year')['track_id'].unique().apply(len).reset_index(), years_df, on='year', how='right').fillna(0)['track_id'].to_numpy()
    bytestorm_iav[basin] = pd.merge(bytestorm[bytestorm['basin']==basin].groupby(by='year')['track_id'].unique().apply(len).reset_index(), years_df, on='year', how='right').fillna(0)['track_id'].to_numpy()

In [ ]:
ibtracs_iav_pearsonr, ibtracs_iav_pearsonr_round, ibtracs_iav_pearsonr_signif = dict(), dict(), dict()
owz_iav_pearsonr, owz_iav_pearsonr_round, owz_iav_pearsonr_signif = dict(), dict(), dict()
track_iav_pearsonr, track_iav_pearsonr_round, track_iav_pearsonr_signif = dict(), dict(), dict()
uz_iav_pearsonr, uz_iav_pearsonr_round, uz_iav_pearsonr_signif = dict(), dict(), dict()
cnrm_iav_pearsonr, cnrm_iav_pearsonr_round, cnrm_iav_pearsonr_signif = dict(), dict(), dict()
bytestorm_iav_pearsonr, bytestorm_iav_pearsonr_round, bytestorm_iav_pearsonr_signif = dict(), dict(), dict()

for basin in basins:
    ibtracs_iav_pearsonr[basin] = pearsonr(ibtracs_iav[basin], ibtracs_iav[basin])
    owz_iav_pearsonr[basin] = pearsonr(ibtracs_iav[basin], owz_iav[basin])
    track_iav_pearsonr[basin] = pearsonr(ibtracs_iav[basin], track_iav[basin])
    uz_iav_pearsonr[basin] = pearsonr(ibtracs_iav[basin], uz_iav[basin])
    cnrm_iav_pearsonr[basin] = pearsonr(ibtracs_iav[basin], cnrm_iav[basin])
    bytestorm_iav_pearsonr[basin] = pearsonr(ibtracs_iav[basin], bytestorm_iav[basin])

    ibtracs_iav_pearsonr_round[basin] = np.round(ibtracs_iav_pearsonr[basin].statistic, 2)
    owz_iav_pearsonr_round[basin] = np.round(owz_iav_pearsonr[basin].statistic, 2)
    track_iav_pearsonr_round[basin] = np.round(track_iav_pearsonr[basin].statistic, 2)
    uz_iav_pearsonr_round[basin] = np.round(uz_iav_pearsonr[basin].statistic, 2)
    cnrm_iav_pearsonr_round[basin] = np.round(cnrm_iav_pearsonr[basin].statistic, 2)
    bytestorm_iav_pearsonr_round[basin] = np.round(bytestorm_iav_pearsonr[basin].statistic, 2)

    ibtracs_iav_pearsonr_signif[basin] = signif(ibtracs_iav_pearsonr[basin].pvalue, 2)
    owz_iav_pearsonr_signif[basin] = signif(owz_iav_pearsonr[basin].pvalue, 2)
    track_iav_pearsonr_signif[basin] = signif(track_iav_pearsonr[basin].pvalue, 2)
    uz_iav_pearsonr_signif[basin] = signif(uz_iav_pearsonr[basin].pvalue, 2)
    cnrm_iav_pearsonr_signif[basin] = signif(cnrm_iav_pearsonr[basin].pvalue, 2)
    bytestorm_iav_pearsonr_signif[basin] = signif(bytestorm_iav_pearsonr[basin].pvalue, 2)

In [ ]:
fig = plt.figure(constrained_layout=True, figsize=(12,4))
gs = fig.add_gridspec(gs_rows, gs_cols)
fig.suptitle(f'Interannual Variability ({years[0]}-{years[-1]})', fontsize=16)
for i, basin in enumerate([b for b in basins if b not in excluded_basins]):
    gsi = gsix(gs,i)
    axi = fig.add_subplot(gsi)
    axi.plot(
        np.arange(len(years)),
        owz_iav[basin],
        label=INFO['owz']['label'],
        linewidth=1.0,
        alpha=0.5,
        color=INFO['owz']['color'],
    )
    axi.plot(
        np.arange(len(years)),
        track_iav[basin],
        label=INFO['track']['label'],
        linewidth=1.0,
        alpha=0.5,
        color=INFO['track']['color'],
    )
    axi.plot(
        np.arange(len(years)),
        uz_iav[basin],
        label=INFO['uz']['label'],
        linewidth=1.0,
        alpha=0.5,
        color=INFO['uz']['color'],
    )
    axi.plot(
        np.arange(len(years)),
        cnrm_iav[basin],
        label=INFO['cnrm']['label'],
        linewidth=1.0,
        alpha=0.5,
        color=INFO['cnrm']['color'],
    )
    axi.plot(
        np.arange(len(years)),
        bytestorm_iav[basin],
        label=INFO['bytestorm']['label'],
        linewidth=1.2,
        color=INFO['bytestorm']['color'],
    )
    axi.plot(
        np.arange(len(years)),
        ibtracs_iav[basin],
        label=INFO['ibtracs']['label'],
        linewidth=1.2,
        linestyle="dashed",
        color=INFO['ibtracs']['color'],
    )

    vmax = np.nanmax((
        ibtracs_iav[basin],
        owz_iav[basin],
        track_iav[basin],
        uz_iav[basin],
        cnrm_iav[basin],
        bytestorm_iav[basin],
    ))

    axi.text(0, vmax, basin, fontsize=12, weight='bold', color='black', ha='left', va='center')
    axi.spines[['right', 'top']].set_visible(False)

    if len(years) >= 20:
        axi.set_xticks(np.arange(len(years), step=5))
        axi.set_xticklabels(years[::5], rotation=45)
    else:
        axi.set_xticks(np.arange(len(years)))
        axi.set_xticklabels(years, rotation=45)

    if i > 3:
        axi.set_xlabel('Year', fontdict={'size':12})
    if i == 0 or i == 3:
        axi.set_ylabel('# TC genesis', fontdict={'size':12})

    plt.ylim(0, None)
    if not i: handles, labels = axi.get_legend_handles_labels()

leg = fig.legend(handles, labels, loc='upper center', markerscale=12, fontsize=10, edgecolor='gray', framealpha=1, ncol=6, bbox_to_anchor=(0.5, 0.02))
for legobj in leg.legend_handles:
    legobj.set_linewidth(2.0)
plt.tight_layout()
outfile = os.path.join(out_dir, f'interannual_variability_by_basin.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
owz_iav_df = pd.concat([pd.DataFrame(data=dict(basin=[basin], pearson=[owz_iav_pearsonr_round[basin]], signif=[owz_iav_pearsonr_signif[basin]])) for basin in basins if basin not in excluded_basins])
owz_iav_df['algo'] = INFO['owz']['label']

track_iav_df = pd.concat([pd.DataFrame(data=dict(basin=[basin], pearson=[track_iav_pearsonr_round[basin]], signif=[track_iav_pearsonr_signif[basin]])) for basin in basins if basin not in excluded_basins])
track_iav_df['algo'] = INFO['track']['label']

uz_iav_df = pd.concat([pd.DataFrame(data=dict(basin=[basin], pearson=[uz_iav_pearsonr_round[basin]], signif=[uz_iav_pearsonr_signif[basin]])) for basin in basins if basin not in excluded_basins])
uz_iav_df['algo'] = INFO['uz']['label']

cnrm_iav_df = pd.concat([pd.DataFrame(data=dict(basin=[basin], pearson=[cnrm_iav_pearsonr_round[basin]], signif=[cnrm_iav_pearsonr_signif[basin]])) for basin in basins if basin not in excluded_basins])
cnrm_iav_df['algo'] = INFO['cnrm']['label']

bytestorm_iav_df = pd.concat([pd.DataFrame(data=dict(basin=[basin], pearson=[bytestorm_iav_pearsonr_round[basin]], signif=[bytestorm_iav_pearsonr_signif[basin]])) for basin in basins if basin not in excluded_basins])
bytestorm_iav_df['algo'] = INFO['bytestorm']['label']

iav_df = pd.concat((owz_iav_df,track_iav_df,uz_iav_df,cnrm_iav_df,bytestorm_iav_df)).reset_index(drop=True)
iav_df.head()

In [ ]:
# iav_df.sort_values('basin').to_csv('table.csv')

In [ ]:
bytestorm_ = bytestorm.copy()
bytestorm_['lon'] = ((bytestorm['lon'] + 360) % 360)
ibtracs_ = ibtracs.copy()
ibtracs_['lon'] = ((ibtracs['lon'] + 360) % 360)
year = 2014
plot_tracks(
    bytestorm_[bytestorm_['time'].dt.year.isin([year])].rename(columns = {'track_id': 'TRACK_ID', 'lat':'LAT', 'lon': 'LON'}), 
    ibtracs_[ibtracs_['time'].dt.year.isin([year])].rename(columns = {'track_id': 'TRACK_ID', 'lat':'LAT', 'lon': 'LON'}), 
    lat_range, 
    lon_range, 
    title='', 
    outfile=os.path.join(out_dir, f'plot_tracks_{our_work_name}_{year}.{file_ext}') if save_figures else None)

# plot_tracks(
#     track[track['time'].dt.year.isin([year])].rename(columns = {'track_id': 'TRACK_ID', 'lat':'LAT', 'lon': 'LON'}), 
#     ibtracs[ibtracs['time'].dt.year.isin([year])].rename(columns = {'track_id': 'TRACK_ID', 'lat':'LAT', 'lon': 'LON'}), 
#     lat_range, 
#     lon_range, 
#     title='', 
#     )

In [ ]:
# from clAImate.cyclone import plot_detections
# bss = pd.read_csv('/Users/davide/Developer/tropical_cyclone_tracking/data/inference-global/bytestorm_loc_NHS-bytestorm_cls_NHS/2014.csv', index_col=0)
# bss['ISO_TIME'] = pd.to_datetime(bss['ISO_TIME'])
# ibs = ibtracs_[ibtracs_['time'].dt.year.isin([year])].rename(columns = {'track_id': 'TRACK_ID', 'lat':'LAT', 'lon': 'LON'}).copy()
# ibs = ibs[(ibs['LON'] >= 290) & (ibs['LAT'] >= 40)]
# dts = ibs.time
# bss = bss[bss.ISO_TIME.isin(dts)]
# year = 2014
# plot_detections(
#     bss, ibs,
#     lat_range, 
#     lon_range, 
#     outfile=None,
# )

In [ ]:
year = 2018
plot_tracks(
    bytestorm_[bytestorm_['time'].dt.year.isin([year])].rename(columns = {'track_id': 'TRACK_ID', 'lat':'LAT', 'lon': 'LON'}), 
    ibtracs_[ibtracs_['time'].dt.year.isin([year])].rename(columns = {'track_id': 'TRACK_ID', 'lat':'LAT', 'lon': 'LON'}), 
    lat_range, 
    lon_range, 
    title='', 
    outfile=os.path.join(out_dir, f'plot_tracks_{our_work_name}_{year}.{file_ext}') if save_figures else None)

# plot_tracks(
#     track[track['time'].dt.year.isin([year])].rename(columns = {'track_id': 'TRACK_ID', 'lat':'LAT', 'lon': 'LON'}), 
#     ibtracs[ibtracs['time'].dt.year.isin([year])].rename(columns = {'track_id': 'TRACK_ID', 'lat':'LAT', 'lon': 'LON'}), 
#     lat_range, 
#     lon_range, 
#     title='', 
#     )

## De-Trended Interannual Variability

In [ ]:
x = np.arange(len(years))
ibtracs_iav_norm, owz_iav_norm, track_iav_norm, uz_iav_norm, cnrm_iav_norm, bytestorm_iav_norm = dict(), dict(), dict(), dict(), dict(), dict()
ibtracs_poly1d_fn, owz_poly1d_fn, track_poly1d_fn, uz_poly1d_fn, cnrm_poly1d_fn, bytestorm_poly1d_fn = dict(), dict(), dict(), dict(), dict(), dict()
ibtracs_iav_pearsonr_norm, owz_iav_pearsonr_norm, track_iav_pearsonr_norm, uz_iav_pearsonr_norm, cnrm_iav_pearsonr_norm, bytestorm_iav_pearsonr_norm = dict(), dict(), dict(), dict(), dict(), dict()
ibtracs_iav_pearsonr_round_norm, owz_iav_pearsonr_round_norm, track_iav_pearsonr_round_norm, uz_iav_pearsonr_round_norm, cnrm_iav_pearsonr_round_norm, bytestorm_iav_pearsonr_round_norm = dict(), dict(), dict(), dict(), dict(), dict()
ibtracs_iav_pearsonr_signif_norm, owz_iav_pearsonr_signif_norm, track_iav_pearsonr_signif_norm, uz_iav_pearsonr_signif_norm, cnrm_iav_pearsonr_signif_norm, bytestorm_iav_pearsonr_signif_norm = dict(), dict(), dict(), dict(), dict(), dict()
for basin in [b for b in basins if b not in excluded_basins]:
    # compute interannual variability
    ibtracs_iav_norm[basin] = pd.merge(ibtracs[ibtracs['basin']==basin].groupby(by='year')['track_id'].unique().apply(len).reset_index(), years_df, on='year', how='right').fillna(0)['track_id'].to_numpy()
    owz_iav_norm[basin] = pd.merge(owz[owz['basin']==basin].groupby(by='year')['track_id'].unique().apply(len).reset_index(), years_df, on='year', how='right').fillna(0)['track_id'].to_numpy()
    track_iav_norm[basin] = pd.merge(track[track['basin']==basin].groupby(by='year')['track_id'].unique().apply(len).reset_index(), years_df, on='year', how='right').fillna(0)['track_id'].to_numpy()
    uz_iav_norm[basin] = pd.merge(uz[uz['basin']==basin].groupby(by='year')['track_id'].unique().apply(len).reset_index(), years_df, on='year', how='right').fillna(0)['track_id'].to_numpy()
    cnrm_iav_norm[basin] = pd.merge(cnrm[cnrm['basin']==basin].groupby(by='year')['track_id'].unique().apply(len).reset_index(), years_df, on='year', how='right').fillna(0)['track_id'].to_numpy()
    bytestorm_iav_norm[basin] = pd.merge(bytestorm[bytestorm['basin']==basin].groupby(by='year')['track_id'].unique().apply(len).reset_index(), years_df, on='year', how='right').fillna(0)['track_id'].to_numpy()

    # compute the trend
    ibtracs_poly1d_fn[basin] = np.poly1d(np.polyfit(x, ibtracs_iav_norm[basin], 1))
    owz_poly1d_fn[basin] = np.poly1d(np.polyfit(x, owz_iav_norm[basin], 1))
    track_poly1d_fn[basin] = np.poly1d(np.polyfit(x, track_iav_norm[basin], 1))
    uz_poly1d_fn[basin] = np.poly1d(np.polyfit(x, uz_iav_norm[basin], 1))
    cnrm_poly1d_fn[basin] = np.poly1d(np.polyfit(x, cnrm_iav_norm[basin], 1))
    bytestorm_poly1d_fn[basin] = np.poly1d(np.polyfit(x, bytestorm_iav_norm[basin], 1))

    # subtract the trend from the variability
    ibtracs_iav_norm[basin] = ibtracs_iav_norm[basin] - ibtracs_poly1d_fn[basin](x)
    owz_iav_norm[basin] = owz_iav_norm[basin] - owz_poly1d_fn[basin](x)
    track_iav_norm[basin] = track_iav_norm[basin] - track_poly1d_fn[basin](x)
    uz_iav_norm[basin] = uz_iav_norm[basin] - uz_poly1d_fn[basin](x)
    cnrm_iav_norm[basin] = cnrm_iav_norm[basin] - cnrm_poly1d_fn[basin](x)
    bytestorm_iav_norm[basin] = bytestorm_iav_norm[basin] - bytestorm_poly1d_fn[basin](x)

    # compute the pearson correlation coefficient
    ibtracs_iav_pearsonr_norm[basin] = pearsonr(ibtracs_iav_norm[basin], ibtracs_iav_norm[basin])
    owz_iav_pearsonr_norm[basin] = pearsonr(ibtracs_iav_norm[basin], owz_iav_norm[basin])
    track_iav_pearsonr_norm[basin] = pearsonr(ibtracs_iav_norm[basin], track_iav_norm[basin])
    uz_iav_pearsonr_norm[basin] = pearsonr(ibtracs_iav_norm[basin], uz_iav_norm[basin])
    cnrm_iav_pearsonr_norm[basin] = pearsonr(ibtracs_iav_norm[basin], cnrm_iav_norm[basin])
    bytestorm_iav_pearsonr_norm[basin] = pearsonr(ibtracs_iav_norm[basin], bytestorm_iav_norm[basin])

    # get the statistics and round it
    ibtracs_iav_pearsonr_round_norm[basin] = np.round(ibtracs_iav_pearsonr_norm[basin].statistic, 2)
    owz_iav_pearsonr_round_norm[basin] = np.round(owz_iav_pearsonr_norm[basin].statistic, 2)
    track_iav_pearsonr_round_norm[basin] = np.round(track_iav_pearsonr_norm[basin].statistic, 2)
    uz_iav_pearsonr_round_norm[basin] = np.round(uz_iav_pearsonr_norm[basin].statistic, 2)
    cnrm_iav_pearsonr_round_norm[basin] = np.round(cnrm_iav_pearsonr_norm[basin].statistic, 2)
    bytestorm_iav_pearsonr_round_norm[basin] = np.round(bytestorm_iav_pearsonr_norm[basin].statistic, 2)

    # get the significance value
    ibtracs_iav_pearsonr_signif_norm[basin] = signif(ibtracs_iav_pearsonr_norm[basin].pvalue, 2)
    owz_iav_pearsonr_signif_norm[basin] = signif(owz_iav_pearsonr_norm[basin].pvalue, 2)
    track_iav_pearsonr_signif_norm[basin] = signif(track_iav_pearsonr_norm[basin].pvalue, 2)
    uz_iav_pearsonr_signif_norm[basin] = signif(uz_iav_pearsonr_norm[basin].pvalue, 2)
    cnrm_iav_pearsonr_signif_norm[basin] = signif(cnrm_iav_pearsonr_norm[basin].pvalue, 2)
    bytestorm_iav_pearsonr_signif_norm[basin] = signif(bytestorm_iav_pearsonr_norm[basin].pvalue, 2)

In [ ]:
fig = plt.figure(constrained_layout=True, figsize=(12,4))
gs = fig.add_gridspec(gs_rows, gs_cols)
fig.suptitle(f'De-Trended Interannual Variability ({years[0]}-{years[-1]})', fontsize=16)
for i, basin in enumerate([b for b in basins if b not in excluded_basins]):
    gsi = gsix(gs, i)
    axi = fig.add_subplot(gsi)
    axi.plot(
        np.arange(len(years)), owz_iav_norm[basin],
        linewidth=1.0,
        alpha=0.5,
        label=INFO['owz']['label'],
        color=INFO['owz']['color'],
    )
    axi.plot(
        np.arange(len(years)), track_iav_norm[basin],
        linewidth=1.0,
        alpha=0.5,
        label=INFO['track']['label'],
        color=INFO['track']['color'],
    )
    axi.plot(
        np.arange(len(years)), uz_iav_norm[basin],
        linewidth=1.0,
        alpha=0.5,
        label=INFO['uz']['label'],
        color=INFO['uz']['color'],
    )
    axi.plot(
        np.arange(len(years)), cnrm_iav_norm[basin],
        linewidth=1.0,
        alpha=0.5,
        label=INFO['cnrm']['label'],
        color=INFO['cnrm']['color'],
    )
    axi.plot(
        np.arange(len(years)), bytestorm_iav_norm[basin],
        linewidth=1.2,
        alpha=1.0,
        label=INFO['bytestorm']['label'],
        color=INFO['bytestorm']['color'],
    )
    axi.plot(
        np.arange(len(years)), ibtracs_iav_norm[basin],
        linewidth=1.5,
        linestyle="dashed",
        alpha=1.0,
        label=INFO['ibtracs']['label'],
        color=INFO['ibtracs']['color'],
    )

    vmax = np.nanmax((
        ibtracs_iav_norm[basin],
        owz_iav_norm[basin],
        track_iav_norm[basin],
        uz_iav_norm[basin],
        cnrm_iav_norm[basin],
        bytestorm_iav_norm[basin],
    ))

    axi.text(0, vmax, basin, fontsize=12, weight='bold', color='black', ha='left', va='center')
    axi.spines[['right', 'top']].set_visible(False)

    if len(years) >= 20:
        axi.set_xticks(np.arange(len(years), step=5))
        axi.set_xticklabels(years[::5], rotation=45)
    else:
        axi.set_xticks(np.arange(len(years)))
        axi.set_xticklabels(years, rotation=45)

    if i > 3:
        axi.set_xlabel('Year', fontdict={'size':12})
    if i == 0 or i == 3:
        axi.set_ylabel('# TC genesis', fontdict={'size':12})

    # plt.ylim(0, None)
    if not i: handles, labels = axi.get_legend_handles_labels()
leg = fig.legend(handles, labels, loc='upper center', markerscale=12, fontsize=10, edgecolor='gray', framealpha=1, ncol=6, bbox_to_anchor=(0.5, 0.02))
for legobj in leg.legend_handles:
    legobj.set_linewidth(2.0)
plt.tight_layout()

outfile = os.path.join(out_dir, f'interannual_variability_by_basin_detrended.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
owz_iav_dt_df = pd.concat([pd.DataFrame(data=dict(basin=[basin], pearson=[owz_iav_pearsonr_round_norm[basin]], signif=[owz_iav_pearsonr_signif_norm[basin]])) for basin in basins if basin not in excluded_basins])
owz_iav_dt_df['algo'] = INFO['owz']['label']

track_iav_dt_df = pd.concat([pd.DataFrame(data=dict(basin=[basin], pearson=[track_iav_pearsonr_round_norm[basin]], signif=[track_iav_pearsonr_signif_norm[basin]])) for basin in basins if basin not in excluded_basins])
track_iav_dt_df['algo'] = INFO['track']['label']

uz_iav_dt_df = pd.concat([pd.DataFrame(data=dict(basin=[basin], pearson=[uz_iav_pearsonr_round_norm[basin]], signif=[uz_iav_pearsonr_signif_norm[basin]])) for basin in basins if basin not in excluded_basins])
uz_iav_dt_df['algo'] = INFO['uz']['label']

cnrm_iav_dt_df = pd.concat([pd.DataFrame(data=dict(basin=[basin], pearson=[cnrm_iav_pearsonr_round_norm[basin]], signif=[cnrm_iav_pearsonr_signif_norm[basin]])) for basin in basins if basin not in excluded_basins])
cnrm_iav_dt_df['algo'] = INFO['cnrm']['label']

bytestorm_iav_dt_df = pd.concat([pd.DataFrame(data=dict(basin=[basin], pearson=[bytestorm_iav_pearsonr_round_norm[basin]], signif=[bytestorm_iav_pearsonr_signif_norm[basin]])) for basin in basins if basin not in excluded_basins])
bytestorm_iav_dt_df['algo'] = INFO['bytestorm']['label'].upper()

iav_dt_df = pd.concat(( owz_iav_dt_df, track_iav_dt_df, uz_iav_dt_df, cnrm_iav_dt_df, bytestorm_iav_dt_df))
iav_dt_df = iav_dt_df.sort_values('basin').reset_index(drop=True)
iav_dt_df.head()

In [ ]:
outfile = os.path.join(out_dir, 'tableII.csv')
iav_dt_df.to_csv(outfile) if save_figures else None

# Additional Plots

In [ ]:
bytestorm_winds = []
for i,row in bytestorm_matches.iterrows():
    tid1, tid2 = row[f'id_{our_work_name}'], row['id_ibtracs']
    merge = pd.merge(bytestorm[bytestorm['track_id']==tid1], ibtracs[ibtracs['track_id']==tid2], on='time')
    merge['dist'] = haversine_vector(merge[['lat_x','lon_x']].to_numpy(), merge[['lat_y','lon_y']].to_numpy(), normalize=True)
    merge = merge[merge['dist'] <= max_track_distance_matching]
    bytestorm_winds += list([(float(w), float(d), n) for w,d,n in zip(merge['WMO_WIND'].values, merge['dist'].values, merge['NATURE'].values) if w != ' '])
bytestorm_df = pd.DataFrame(
    data={'MSW': [w for w,_,_ in bytestorm_winds], 'DIST': [d for _,d,_ in bytestorm_winds], 'NATURE': [n for _,_,n in bytestorm_winds]}
)

In [ ]:
natures = ['TS', 'DS', 'MX', 'NR', 'ET']
colors = ['darkblue', 'tab:green', 'tab:purple', 'tab:red', 'tab:cyan']
plt.figure(figsize=(8,6))
for nature, color in zip(natures, colors):
    nature_df = bytestorm_df[bytestorm_df.NATURE==nature]
    alpha = 0.8 if nature in ['ET', 'DS', 'MX', 'NR'] else 0.5
    plt.plot(nature_df['DIST'], nature_df['MSW'], label=nature, linestyle='', marker='o', markersize=6, alpha=alpha)
plt.legend()
plt.xlabel('Distance [km]')
plt.ylabel('MSW [kn]')
outfile = os.path.join(out_dir, f'dist_msw_plot_1.{file_ext}') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

In [ ]:
plt.figure(figsize=(8,6))
sns.jointplot(bytestorm_df, x='DIST', y='MSW', kind='hist')
plt.xlabel('Distance [km]')
plt.ylabel('MSW [kn]')
outfile = os.path.join(out_dir, f'dist_msw_plot_2.{file_ext}') if save_figures else None
plt.savefig(outfile, **plot_kwargs) if outfile else plt.show()

## Answering some Questions

In [ ]:
def clean_ibtracs_dirty(ib):
    # All about wind
    ## Select from the data & convert
    ib["WIND10"] = np.where(
        ~ib.WMO_WIND.isna(),
        ib.WMO_WIND,
        ib[
            ["TOKYO_WIND", "REUNION_WIND", "BOM_WIND", "NADI_WIND", "WELLINGTON_WIND"]
        ].mean(axis=1, skipna=True),
    )
    ib["WIND10"] = np.where(
        ib.WIND10.isna(), ib.USA_WIND / 1.12, ib.WIND10
    )  # Conversion rate in the IBTrACS doc
    ib["WIND10"] = np.where(
        ib.WIND10.isna(), ib.CMA_WIND / 1.08, ib.WIND10
    )  # Conversion rate determined through a linear regression
    ib["WIND10"] *= 0.514  # Conversion noeuds en m/s

    # Select pressure
    ib["PRES"] = np.where(
        ~ib.WMO_PRES.isna(),
        ib.WMO_PRES,
        ib[
            [
                "USA_PRES",
                "TOKYO_PRES",
                "CMA_PRES",
                "HKO_PRES",
                "NEWDELHI_PRES",
                "REUNION_PRES",
                "BOM_PRES",
                "NADI_PRES",
                "WELLINGTON_PRES",
            ]
        ].mean(axis=1, skipna=True),
    )
    return ib

In [ ]:
ibd_src = '/work/cmcc/machine_learning/dd26322/tropical_cyclone_tracking/data/ibtracs/full/new-ibtracs.ALL.list.v04r01.csv'
ibd = pd.read_csv(ibd_src, na_values=["", " "])
ibd = ibd.rename(
    columns={'ISO_TIME':'time','LAT':'lat','LON':'lon','SID':'track_id','BASIN':'basin'}
) # [['time','lat','lon','track_id','basin','NATURE','WMO_WIND', 'TRACK_TYPE']]
ibd['time'] = pd.to_datetime(ibd['time'])
print(f'Total {len(ibd)}')
ibd = ibd[ibd['time'].isin(dates)]
print(f'Date removal {len(ibd)}')
ibd['basin'] = dynamicopy.get_basin(ibd['lon'], ibd['lat'])
coords_format = '[-180,180]'
ibd = convert_coords_to_format(ibd, format=coords_format)
ibd = clean_ibtracs_dirty(ibd)
print(f'Cleaning {len(ibd)}')
ibd = ibd.reset_index(drop=True)
ibd.head()

In [ ]:
def remove_long_tracks(tracks, max_track_duration):
    return tracks[tracks['track_id'].isin(tracks.groupby('track_id').filter(lambda x: len(x) <= max_track_duration)['track_id'].unique())].reset_index(drop=True)

### How are distributed ByteStorm's False Alarms?

In [ ]:
bytestorm_fa_matches_dirty, bytestorm_fa_res_dirty = compute_pod_and_far(dynamicopy, bytestorm_false_alarms, our_work_name, ibd, max_track_distance_matching, False)

In [ ]:
merged = pd.merge(bytestorm_fa_matches_dirty, ibd, left_on='id_ibtracs', right_on='track_id')
merged.TRACK_TYPE.unique()

In [ ]:
merged['WIND10'].median(), merged['PRES'].median()

In [ ]:
total_FA = len(bytestorm_false_alarms.track_id.unique())
matched_FA = len(bytestorm_fa_matches_dirty[f'id_{our_work_name}'].unique())
matched_FA, total_FA, matched_FA / total_FA * 100

In [ ]:
bytestorm_fa_matches_dirty

In [ ]:
fig,ax = plt.subplots(figsize=(8,4))
sns.countplot(dynamicopy.genesis_points(bytestorm_false_alarms), x='basin', order=basins, palette=sns.color_palette('deep'), ax=ax)
ax.set_ylabel('# TC Tracks')
ax.set_xlabel('Basin')
ax.spines[['right', 'top']].set_visible(False)
outfile = os.path.join(out_dir, f'fa_all_tc_distribution_by_basin.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

### How are distributed very short tracks (i.e., lasting exactly 3 days)?

In [ ]:
bytestorm_fa_matches_dirty, bytestorm_fa_res_dirty = compute_pod_and_far(dynamicopy, remove_long_tracks(bytestorm_false_alarms, 12), our_work_name, ibd, max_track_distance_matching, False)
total_FA = len(bytestorm_false_alarms.track_id.unique())
matched_FA = len(bytestorm_fa_matches_dirty[f'id_{our_work_name}'].unique())
print(matched_FA, total_FA, matched_FA / total_FA * 100)

fig,ax = plt.subplots(figsize=(8,4))
sns.countplot(dynamicopy.genesis_points(remove_long_tracks(bytestorm_false_alarms, 12)), x='basin', order=basins, palette=sns.color_palette('deep'), ax=ax)
ax.set_ylabel('# TC Tracks')
ax.set_xlabel('Basin')
ax.spines[['right', 'top']].set_visible(False)
outfile = os.path.join(out_dir, f'fa_3_days_long_tc_distribution_by_basin.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

### Which is the smoothness of these False Alarms?

In [ ]:
smoothness_fa = pd.DataFrame(
    columns = ['algo', 'score', 'delta_angles']
)

for track_id in bytestorm_false_alarms.track_id.unique():
    latlon = bytestorm_false_alarms[bytestorm_false_alarms['track_id']==track_id][['lat','lon']].to_numpy()
    lat, lon = latlon[:,0], latlon[:,1]
    score, delta_angles = track_smoothness(lat, lon)
    smoothness_fa = pd.concat([
        smoothness_fa, 
        pd.DataFrame({
            'algo': [our_work_name], 
            'track_id': [track_id], 
            'score': [score], 
            'delta_angles': [delta_angles], 
        })
    ])
smoothness_fa = smoothness_fa.reset_index(drop=True)

In [ ]:
smoothness_tp = pd.DataFrame(
    columns = ['algo', 'score', 'delta_angles']
)

for track_id in bytestorm_true_positives.track_id.unique():
    latlon = bytestorm_true_positives[bytestorm_true_positives['track_id']==track_id][['lat','lon']].to_numpy()
    lat, lon = latlon[:,0], latlon[:,1]
    score, delta_angles = track_smoothness(lat, lon)
    smoothness_tp = pd.concat([
        smoothness_tp, 
        pd.DataFrame({
            'algo': [our_work_name], 
            'track_id': [track_id], 
            'score': [score], 
            'delta_angles': [delta_angles], 
        })
    ])
smoothness_tp = smoothness_tp.reset_index(drop=True)

In [ ]:
smoothness_tp['score'].median(), smoothness_fa['score'].median()

### True positives and False alarms distribution

In [ ]:
bytestorm_true_positives['label'] = 'TP'
bytestorm_false_alarms['label'] = 'FA'
fig,ax = plt.subplots(figsize=(6,4))
sns.countplot(pd.concat((bytestorm_false_alarms, bytestorm_true_positives)), x='basin', hue='label', order=basins, hue_order=['TP','FA'], palette=['tab:blue','tab:red'], ax=ax)
ax.legend()
ax.set_ylabel('# TC Detections')
ax.set_xlabel('Basin')
ax.spines[['right', 'top']].set_visible(False)
outfile = os.path.join(out_dir, f'tp_fa_distribution_by_basin.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
ibtracs['algo']   = INFO['ibtracs']['label']
owz['algo']       = INFO['owz']['label']
track['algo']     = INFO['track']['label']
uz['algo']        = INFO['uz']['label']
cnrm['algo']      = INFO['cnrm']['label']
bytestorm['algo'] = INFO['bytestorm']['label']

palette = [
    INFO['ibtracs']['color'],
    INFO['owz']['color'],
    INFO['track']['color'],
    INFO['uz']['color'],
    INFO['cnrm']['color'],
    INFO['bytestorm']['color'],
]

all_trackers_df = pd.concat((
    ibtracs, owz, track, uz, cnrm, bytestorm,
)).reset_index(drop=True)
fig,ax = plt.subplots(figsize=(8,4))
sns.countplot(all_trackers_df, x='basin', hue='algo', order=basins, palette=palette, ax=ax)
ax.set_ylabel('# TC Detections', fontdict=dict(size=12))
ax.set_xlabel('Basin', fontdict=dict(size=12))
ax.set_ylim([0,9000])
ax.spines[['right', 'top']].set_visible(False)
leg = ax.legend(loc='upper center', markerscale=12, fontsize=10, edgecolor='gray', framealpha=1, ncol=6, bbox_to_anchor=(0.5, -0.14))
for legobj in leg.legend_handles:
    legobj.set_linewidth(2.0)
outfile = os.path.join(out_dir, f'tc_distribution_by_basin.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()

In [ ]:
cols = ['basin', 'algo', 'hits', 'misses', 'false alarms', 'pod', 'far', 'sr', 'csi', 'bias', 'pearson', 'signif']
table_III = pd.merge(
    pd.read_csv(os.path.join(out_dir, 'tableI.csv'), index_col=0),
    pd.read_csv(os.path.join(out_dir, 'tableII.csv'), index_col=0),
    on=['algo', 'basin']
)[cols].reset_index(drop=True)
table_III.to_csv(os.path.join(out_dir, 'tableIII.csv')) if save_figures else table_III

In [ ]:
import matplotlib.gridspec as gridspec

In [ ]:
bs_true_fas = bytestorm_false_alarms[~bytestorm_false_alarms.track_id.isin(bytestorm_fa_matches_dirty[f'id_{our_work_name}'].unique())]
n = bs_true_fas.track_id.unique().shape[0]
tids = bs_true_fas.track_id.unique()[np.random.randint(0, n, 10)]
# convert_coords_to_format(bytestorm_false_alarms, '[0,360]')
tids

In [ ]:
# len(bytestorm_false_alarms.track_id.unique()), len(bytestorm_false_alarms)
# nrows = int(np.floor(np.sqrt(1985)))
# ncols = int(np.ceil(1985 / nrows))
# nrows, ncols

In [ ]:
from tqdm import tqdm

In [ ]:
vo_imgs = []
msl_imgs = []
data_dir = '/work/cmcc/machine_learning/dd26322/tropical_cyclone_tracking/data/datasets/global'
iterable = bytestorm_false_alarms.iterrows()
for i,row in tqdm(iterable, total=len(bytestorm_false_alarms)):
    lat, lon, date = row['lat'], row['lon'], row['time']
    fname = os.path.join(data_dir, f'{date.year}_{date.month:02d}_{date.day:02d}_{date.hour:02d}.nc')
    ds = xr.open_dataset(fname).sel(lat=slice(lat+5, lat-5), lon=slice(lon-5, lon+5)).isel(time=0)
    if ds.lat.shape[0] == 0: 
        # print(f'skip {fname}')
        continue
    if ds.lon.shape[0] == 0: 
        # print(f'skip {fname}')
        continue
    vo_imgs.append(ds['vo_850'].values)
    msl_imgs.append(ds['msl'].values)

In [ ]:
print(len(vo_imgs), len(msl_imgs))
remove_ids = [382, 383, 384, 385, 386, 387, 388, 389, 390, 391, 392]
vo_imgs = [im for i,im in enumerate(vo_imgs) if i not in remove_ids]
msl_imgs = [im for i,im in enumerate(msl_imgs) if i not in remove_ids]
print(len(vo_imgs), len(msl_imgs))

In [ ]:
# len(bytestorm_false_alarms.track_id.unique()), len(bytestorm_false_alarms)
assert len(vo_imgs) == len(msl_imgs)
nims = len(vo_imgs)
nrows = int(np.floor(np.sqrt(nims)))
ncols = int(np.ceil(nims / nrows))

# vo_vmin = np.min(np.stack(vo_imgs))
# vo_vmax = np.max(np.stack(vo_imgs))
# msl_vmin = np.min(np.stack(msl_imgs))
# msl_vmax = np.max(np.stack(msl_imgs))

cmap = 'RdBu_r'
fig = plt.figure(figsize=(nrows*2, ncols))
gs = gridspec.GridSpec(nrows=nrows, ncols=ncols, figure=fig, wspace=0.1, hspace=0.0)
plt.axis('off')
k = 0
for i in range(nrows):
    for j in range(ncols):
        try:
            gsi = gridspec.GridSpecFromSubplotSpec(1,2, subplot_spec=gs[i,j], wspace=0.0, hspace=0.0)
            axi = fig.add_subplot(gsi[0,0])
            im = axi.imshow(
                vo_imgs[k], 
                cmap=cmap, 
                # vmin=vo_vmin, 
                # vmax=vo_vmax, 
            )
            axi.axis('off')
            axi = fig.add_subplot(gsi[0,1])
            im = axi.imshow(
                msl_imgs[k], 
                cmap=cmap, 
                # vmin=msl_vmin, 
                # vmax=msl_vmax, 
            )
            axi.axis('off')
        except:
            gsi = gridspec.GridSpecFromSubplotSpec(1,2, subplot_spec=gs[i,j], wspace=0.0, hspace=0.0)
            axi = fig.add_subplot(gsi[0,0])
            axi.axis('off')
            axi = fig.add_subplot(gsi[0,1])
            axi.axis('off')
        k+=1
plt.tight_layout()
outfile = os.path.join(out_dir, f'evolution_of_all_false_alarm.{file_ext}')
plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()
# plt.show()

In [ ]:
# bytestorm_false_alarms_sp = bytestorm_false_alarms[bytestorm_false_alarms['basin']=='SP']
# bs_true_fas_sp = bytestorm_false_alarms_sp[~bytestorm_false_alarms_sp.track_id.isin(bytestorm_fa_matches_dirty[f'id_{our_work_name}'].unique())]
# n_sp = bs_true_fas_sp.track_id.unique().shape[0]
# tids_sp = bs_true_fas_sp.track_id.unique()[np.random.randint(0, n_sp, 10)]
# tids_sp

# cmap='RdBu_r'
# data_dir = '/work/cmcc/machine_learning/dd26322/tropical_cyclone_tracking/data/datasets/global'

# for tid in tids_sp[-1:]:
#     bytestorm_subset = bytestorm_false_alarms[bytestorm_false_alarms.track_id==tid]
#     n_points = len(bytestorm_subset)
#     nrows, ncols = int(np.ceil(n_points/4)), 4
#     if tid in ['OT_47708_(45089-45100)', 'OT_38477_(36344-36358)']:
#         fig = plt.figure(figsize=(nrows*3, ncols))
#     elif tid in ['OT_34517_(32605-32626)']:
#         fig = plt.figure(figsize=(nrows, ncols))
#     elif tid in ['OT_7904_(8206-8240)']:
#         fig = plt.figure(figsize=(nrows/2, ncols))
#     else:
#         fig = plt.figure(figsize=(nrows*2, ncols))
       
#     gs = gridspec.GridSpec(nrows=nrows, ncols=ncols, figure=fig, wspace=0.1, hspace=0.0)
#     lat_min, lat_max = bytestorm_subset.lat.min(), bytestorm_subset.lat.max()
#     lon_min, lon_max = bytestorm_subset.lon.min(), bytestorm_subset.lon.max()
#     # plt.title(f'Evolution of {tid} False Alarm\n(LAT[{lat_min}, {lat_max}], LON[{lon_min}, {lon_max}])')
#     plt.axis('off')
#     for i,row in bytestorm_subset.reset_index(drop=True).iterrows():
#         r, c = i // ncols, i % ncols
#         lat, lon, date = row['lat'], row['lon'], row['time']
#         fname = os.path.join(data_dir, f'{date.year}_{date.month:02d}_{date.day:02d}_{date.hour:02d}.nc')
#         ds = xr.open_dataset(fname).sel(lat=slice(lat+5, lat-5), lon=slice(lon-5, lon+5)).isel(time=0)
#         gsi = gridspec.GridSpecFromSubplotSpec(1,2,subplot_spec=gs[r,c], wspace=0.0, hspace=0.0)
#         axi = fig.add_subplot(gsi[0,0])
#         im = axi.imshow(ds['vo_850'].values, cmap=cmap)
#         axi.axis('off')
#         axi = fig.add_subplot(gsi[0,1])
#         # if i == 0 or i == len(bytestorm_subset)-1:
#         #     axi.text(-38, -1, f'{date.year}-{date.month:02d}-{date.day:02d} {date.hour:02d}:00')
#         im = axi.imshow(ds['msl'].values, cmap=cmap)
#         axi.axis('off')
#     plt.tight_layout()
#     outfile = os.path.join(out_dir, f'evolution_false_alarm_{tid}-SP-BASIN.{file_ext}')
#     plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()
#     # plt.show()

In [ ]:
# tids = [
#     'OT_12947_(13129-13141)', 
#     'OT_13979_(14146-14157)', 
#     'OT_42511_(40353-40377)', 
# ]
# for tid in tids:
#     b = bytestorm_false_alarms[bytestorm_false_alarms.track_id==tid]
#     print(tid, b.time.min(), b.time.max(), b.lat.min(), b.lat.max(), b.lon.min(), b.lon.max())

In [ ]:
cmap='RdBu_r'
data_dir = '/work/cmcc/machine_learning/dd26322/tropical_cyclone_tracking/data/datasets/global'

default_tids = [
    # 'OT_15983_(15837-15851)',
    # 'OT_37331_(35308-35326)',
    # 'OT_23337_(22441-22456)',
    # 'OT_12947_(13129-13141)',
    # 'OT_50981_(48049-48064)',
    'OT_13979_(14146-14157)',
    # 'OT_42511_(40353-40377)',
]

for tid in default_tids:
    bytestorm_subset = bytestorm_false_alarms[bytestorm_false_alarms.track_id==tid]
    n_points = len(bytestorm_subset)
    nrows, ncols = 4, int(np.ceil(n_points/4))
    # if tid in ['OT_152967_(47586-47600)', 'OT_128877_(40357-40387)']:
    #     fig = plt.figure(figsize=(5,3))
    # elif tid in ['OT_144627_(45102-45113)']:
    #     fig = plt.figure(figsize=(6,3))
    # elif tid in ['OT_39506_(13071-13082)']:
    #     fig = plt.figure(figsize=(7,3))
    # elif tid in ['OT_39597_(13094-13125)']:
    #     fig = plt.figure(figsize=(4,5))
    if tid in ['OT_13979_(14146-14157)']:
        fig = plt.figure(figsize=(nrows, ncols))
    else:
        fig = plt.figure(figsize=(nrows*2, ncols))
    gs = gridspec.GridSpec(nrows=nrows, ncols=ncols, figure=fig, wspace=0.1, hspace=0.0)
    lat_min, lat_max = bytestorm_subset.lat.min(), bytestorm_subset.lat.max()
    lon_min, lon_max = bytestorm_subset.lon.min(), bytestorm_subset.lon.max()
    # plt.title(f'Evolution of {tid} False Alarm\n(LAT[{lat_min}, {lat_max}], LON[{lon_min}, {lon_max}])')
    plt.axis('off')
    for i,row in bytestorm_subset.reset_index(drop=True).iterrows():
        r, c = i % nrows, i // nrows
        lat, lon, date = row['lat'], row['lon'], row['time']
        fname = os.path.join(data_dir, f'{date.year}_{date.month:02d}_{date.day:02d}_{date.hour:02d}.nc')
        ds = xr.open_dataset(fname).sel(lat=slice(lat+5, lat-5), lon=slice(lon-5, lon+5)).isel(time=0)
        gsi = gridspec.GridSpecFromSubplotSpec(1,2,subplot_spec=gs[r,c], wspace=0.0, hspace=0.0)
        axi = fig.add_subplot(gsi[0,0])
        im = axi.imshow(ds['vo_850'].values, cmap=cmap)
        axi.axis('off')
        axi = fig.add_subplot(gsi[0,1])
        # if i == 0 or i == len(bytestorm_subset)-1:
        #     axi.text(-38, -1, f'{date.year}-{date.month:02d}-{date.day:02d} {date.hour:02d}:00')
        im = axi.imshow(ds['msl'].values, cmap=cmap)
        axi.axis('off')
    plt.tight_layout()
    outfile = os.path.join(out_dir, f'evolution_false_alarm_{tid}.{file_ext}')
    plt.savefig(outfile, **plot_kwargs) if save_figures else plt.show()
    # plt.show()
    # break

In [ ]:
cmap='RdBu_r'
data_dir = '/work/cmcc/machine_learning/dd26322/tropical_cyclone_tracking/data/datasets/global'

default_tids = [
    'OT_144627_(45102-45113)', 'OT_152967_(47586-47600)',
    'OT_39506_(13071-13082)','OT_39597_(13094-13125)',
    'OT_78398_(24639-24659)','OT_37460_(12456-12478)',
    'OT_113268_(35317-35341)', 'OT_128877_(40357-40387)',
]

for tid in default_tids:
    bytestorm_subset = bytestorm_false_alarms[bytestorm_false_alarms.track_id==tid]
    n_points = len(bytestorm_subset)
    nrows, ncols = int(np.ceil(n_points/4)), 4
    lat_min, lat_max = bytestorm_subset.lat.min(), bytestorm_subset.lat.max()
    lon_min, lon_max = bytestorm_subset.lon.min(), bytestorm_subset.lon.max()
    print(f'tid {tid} LAT[{lat_min}, {lat_max}], LON[{lon_min}, {lon_max}]) Time {bytestorm_subset.sort_values(by="time")["time"].values[0]} {bytestorm_subset.sort_values(by="time")["time"].values[-1]}')